# NexaTel Customer Churn Analytics
## Phase 1 - Data Cleaning & Quality Check

# Objective
The purpose of this phase is to load, inspect, profile, clean and validate the NexaTel datasets before performing exploratory data analysis and churn KPI calculations.

# Dataset
The project contains 24 interconnected CSV tables covering customers, subscriptions, billing, payments, usage, network quality, complaints, support, campaigns and geographical information.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

## Define the Raw Data Location

The original NexaTel CSV files are stored in the project's `data/raw` directory.

A dedicated path variable is created so that the datasets can be accessed consistently throughout the notebook. The raw files will be treated as the original source data and will not be modified directly.

Any cleaned versions produced later in the project will be stored separately in the `data/cleaned` directory.

In [3]:
raw_data_path = Path("../data/raw")

raw_data_path

WindowsPath('../data/raw')

In [4]:
csv_files = list(raw_data_path.glob("*.csv"))

len(csv_files)

24

In [5]:
for file in csv_files:
    print(file.name)

billing.csv
cities.csv
complaints.csv
contracts.csv
customers.csv
customer_feedback.csv
data_quality_issue_log.csv
devices.csv
employees.csv
marketing_campaigns.csv
network_quality.csv
payments.csv
plans.csv
plan_history.csv
recharges.csv
regions.csv
retention_campaigns.csv
states.csv
stores.csv
subscriptions.csv
support_tickets.csv
usage_data.csv
usage_sms.csv
usage_voice.csv


## Load the NexaTel Datasets

All 24 CSV files are loaded into Pandas DataFrames and stored in a Python dictionary called `data`.

The filename is used as the dictionary key, allowing each table to be accessed using its table name, for example:

`data["customers"]`

This approach is more efficient and scalable than writing a separate `pd.read_csv()` statement for every table.

In [6]:
data = {}

for file in csv_files:
    table_name = file.stem
    data[table_name] = pd.read_csv(file)

len(data)

24

In [7]:
data.keys()

dict_keys(['billing', 'cities', 'complaints', 'contracts', 'customers', 'customer_feedback', 'data_quality_issue_log', 'devices', 'employees', 'marketing_campaigns', 'network_quality', 'payments', 'plans', 'plan_history', 'recharges', 'regions', 'retention_campaigns', 'states', 'stores', 'subscriptions', 'support_tickets', 'usage_data', 'usage_sms', 'usage_voice'])

THE CODE IN THE CELL BELOW TRUNCATES THE OUTPUT AUTOMATICALLY SO TO PRINT THE COMPLETE OUTPUT, WE NEED TO WRITE AS FOLLOWS:

FOR PRINTING ALL THE COLUMNS, pd.set_option("display.max_columns", None)

FOR PRINTING ALL THE ROWS, pd.set_option("display.max_rows", None)

In [8]:
pd.set_option("display.max_columns", None)
data["customers"].head()

,customer_id,first_name,last_name,gender,date_of_birth,age,email,phone_number,address,city_id,city_name,state_id,pin_code,city_tier,region_id,occupation,annual_income_inr,customer_segment,product_line,plan_id,acquisition_channel,acquisition_date,tenure_months,customer_status,churn_date
0,C0000001,Harish,Shetty,Male,31-03-1988,38,harish.shetty909@yahoo.in,916647481156,"Plot 382, Tilak Lane, Raipur",CT0026,Raipur,ST027,492680,Tier-2,RG04,School Teacher,791058.0,Mass,Prepaid Mobile,PL002,Retail Store,15-01-2026,2,Churned,22-03-2026
1,C0000002,Anjali,Khan,Female,31-03-1974,52,anjali.khan525@rediffmail.com,918116536619,"Door No. 269, Civil Lines Colony, Gurugram",CT0009,Gurugram,ST002,122535,Metro,RG01,Small Business Owner,1385499.0,Premium,Fiber Broadband,PL017,Retail Store,01-01-2014,146,Active,NaN
2,C0000003,Arjun,Hegde,Male,31-03-1978,48,arjun.hegde378@hotmail.com,918958153134,"House No. 320, Garden Nagar, Vadodara",CT0020,Vadodara,ST011,390978,Tier-2,RG02,Auto/Cab Driver,394773.0,Enterprise,Enterprise,PL021,Tele-sales,11-11-2015,1,Churned,22-12-2015
3,C0000004,Vidya,Parmar,Female,31-03-1988,38,vidya.parmar387@hotmail.com,917763920098,"House No. 204, Industrial Street, Nagpur",CT0015,Nagpur,ST010,440881,Tier-2,RG02,Operations Manager,1977532.0,Premium,Prepaid Mobile,PL006,Tele-sales,2014/10/18,137,Active,NaN
4,C0000005,Arjun,Saxena,Male,31-03-1988,38,arjun.saxena410@yahoo.in,916787478273,"Flat 156, Krishna Sector, Warangal",CT0043,Warangal,ST019,506582,Tier-3,RG03,Retired,847300.0,Value,Prepaid Mobile,PL003,Online,20-10-2014,108,Churned,22-10-2023


## Dataset Structure and Initial Profiling

The next step is to perform an initial structural profile of all 24 datasets.

For each table, the following characteristics are examined:

- Number of rows
- Number of columns
- Object/text columns
- Numeric columns
- Datetime columns

This provides an overview of the size and structure of the database and helps identify potential data-type issues that may require attention during the cleaning process.

The profiling results will also help determine which tables and columns require more detailed validation.

In [9]:
inventory = pd.DataFrame([
    {
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    }
    for name, df in data.items()
])

inventory = inventory.sort_values("table").reset_index(drop=True)

inventory

,table,rows,columns
0,billing,115313,9
1,cities,68,8
2,complaints,16584,7
3,contracts,10607,9
4,customer_feedback,28499,7
5,customers,19076,25
6,data_quality_issue_log,20,3
7,devices,19000,8
8,employees,800,9
9,marketing_campaigns,40,9


In [10]:
for name, df in data.items():
    print(f"\n{'=' * 60}")
    print(f"TABLE: {name}")
    print(f"{'=' * 60}")
    print(df.dtypes)


TABLE: billing
invoice_id                  str
customer_id                 str
billing_date                str
billing_period_month        str
base_amount_inr         float64
gst_amount_inr          float64
total_amount_inr        float64
due_date                    str
payment_status              str
dtype: object

TABLE: cities
city_id         str
city_name       str
state_id        str
state_name      str
zone            str
region_id       str
tier            str
pin_prefix    int64
dtype: object

TABLE: complaints
complaint_id       str
customer_id        str
complaint_date     str
complaint_type     str
severity           str
status             str
resolution_date    str
dtype: object

TABLE: contracts
contract_id                 str
customer_id                 str
plan_id                     str
contract_type               str
contract_length_months    int64
start_date                  str
end_date                    str
auto_renew                  str
renewal_status           

In [11]:
dtype_summary = pd.DataFrame([
    {
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "object_columns": df.select_dtypes(include="object").shape[1],
        "numeric_columns": df.select_dtypes(include=np.number).shape[1],
        "datetime_columns": df.select_dtypes(include="datetime").shape[1]
    }
    for name, df in data.items()
])

dtype_summary.sort_values("table").reset_index(drop=True)

,table,rows,columns,object_columns,numeric_columns,datetime_columns
0,billing,115313,9,6,3,0
1,cities,68,8,7,1,0
2,complaints,16584,7,7,0,0
3,contracts,10607,9,8,1,0
4,customer_feedback,28499,7,5,2,0
5,customers,19076,25,20,5,0
6,data_quality_issue_log,20,3,2,1,0
7,devices,19000,8,6,2,0
8,employees,800,9,9,0,0
9,marketing_campaigns,40,9,6,3,0


## Missing Value Analysis

Missing values are an important part of data-quality assessment because incomplete records can affect churn calculations, customer segmentation and downstream analysis.

The missing-value analysis calculates both:

- The number of missing values in each column
- The percentage of records affected

Only columns containing missing values are highlighted so that attention can be focused on fields that require investigation.

Missing values will be investigated first and handled later according to the business meaning of each column rather than being removed automatically.

In [12]:
pd.set_option("display.max_rows", None)
missing_report = pd.concat([
    pd.DataFrame({
        "table": name,
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_percentage": (df.isna().sum().values / len(df) * 100)
    })
    for name, df in data.items()
], ignore_index=True)

missing_report = missing_report.sort_values(
    ["missing_count", "table"],
    ascending=[False, True]
).reset_index(drop=True)

missing_report

,table,column,missing_count,missing_percentage
0,subscriptions,end_date,17017,82.916728
1,customers,churn_date,15560,81.568463
2,support_tickets,resolution_hours,13366,42.445221
3,support_tickets,resolution_date,13366,42.445221
4,complaints,resolution_date,7348,44.307767
5,customers,annual_income_inr,954,5.001048
6,customers,email,745,3.905431
7,customers,address,380,1.992032
8,subscriptions,plan_id,41,0.199776
9,billing,invoice_id,0,0.000000


In [13]:
missing_only = missing_report[
    missing_report["missing_count"] > 0
].reset_index(drop=True)

missing_only

,table,column,missing_count,missing_percentage
0,subscriptions,end_date,17017,82.916728
1,customers,churn_date,15560,81.568463
2,support_tickets,resolution_hours,13366,42.445221
3,support_tickets,resolution_date,13366,42.445221
4,complaints,resolution_date,7348,44.307767
5,customers,annual_income_inr,954,5.001048
6,customers,email,745,3.905431
7,customers,address,380,1.992032
8,subscriptions,plan_id,41,0.199776


## Duplicate Record Analysis

Duplicate records can lead to incorrect customer counts, inflated revenue figures and distorted churn metrics.

The project requirements specifically identify duplicate keys in important tables such as:

- `customers.customer_id`
- `billing.invoice_id`

Duplicate checks are therefore performed at both the **complete-row level** and the **key level**.

At this stage, the duplicates are only being identified and quantified. No records will be removed until their nature and impact have been investigated.

In [14]:
duplicate_report = pd.DataFrame([
    {
        "table": name,
        "duplicate_rows": df.duplicated().sum(),
        "duplicate_percentage": df.duplicated().sum() / len(df) * 100
    }
    for name, df in data.items()
])

duplicate_report = duplicate_report.sort_values(
    "duplicate_rows",
    ascending=False
).reset_index(drop=True)

duplicate_report

,table,duplicate_rows,duplicate_percentage
0,billing,344,0.298318
1,payments,276,0.298824
2,customers,76,0.398406
3,complaints,0,0.000000
4,contracts,0,0.000000
5,customer_feedback,0,0.000000
6,data_quality_issue_log,0,0.000000
7,cities,0,0.000000
8,devices,0,0.000000
9,employees,0,0.000000


In [15]:
customers = data["customers"]

customers["customer_id"].duplicated().sum()

np.int64(76)

THE OUTPUT OF THE CODE BELOW GENERATED WAY TOO MANY NUMBER OF ROWS SO TO JUST PRINT THE FIRST 20 ROWS WE ADDED .head(20)

In [16]:
customers[
    customers["customer_id"].duplicated(keep=False)
].sort_values("customer_id").head(20)


,customer_id,first_name,last_name,gender,date_of_birth,age,email,phone_number,address,city_id,city_name,state_id,pin_code,city_tier,region_id,occupation,annual_income_inr,customer_segment,product_line,plan_id,acquisition_channel,acquisition_date,tenure_months,customer_status,churn_date
97,C0000098,Ayaan,Sheikh,Male,31-03-1977,49,ayaan.sheikh13@rediffmail.com,917063719000,"House No. 229, Old Phase, Siliguri",CT0052,SILIGURI,ST022,734945,Tier-3,RG04,School Teacher,538733.0,Premium,5G,PL013,Retail Store,04-02-2021,61,Active,NaN
19044,C0000098,Ayaan,Sheikh,Male,31-03-1977,49,ayaan.sheikh13@rediffmail.com,917063719000,"House No. 229, Old Phase, Siliguri",CT0052,SILIGURI,ST022,734945,Tier-3,RG04,School Teacher,538733.0,Premium,5G,PL013,Retail Store,04-02-2021,61,Active,NaN
19074,C0000169,Yash,Shinde,Male,01-04-1971,55,yash.shinde18@outlook.com,918905657883,"Flat 66, Lake Nagar, Belagavi",CT0041,Belagavi,ST015,590347,Tier-3,RG03,Sales Executive,506678.0,Value,Prepaid Mobile,PL003,Retail Store,08-11-2020,64,Active,NaN
168,C0000169,Yash,Shinde,Male,01-04-1971,55,yash.shinde18@outlook.com,918905657883,"Flat 66, Lake Nagar, Belagavi",CT0041,Belagavi,ST015,590347,Tier-3,RG03,Sales Executive,506678.0,Value,Prepaid Mobile,PL003,Retail Store,08-11-2020,64,Active,NaN
19025,C0000784,Sita,Hussain,Female,31-03-2008,18,sita.hussain386@rediffmail.com,917836729786,"Apartment 26, Garden Colony, Shimla",CT0061,Shimla,ST007,5845,Tier-3,RG01,Self Employed,626862.0,Mass,Prepaid Mobile,PL002,Retail Store,27-09-2018,90,Active,NaN
783,C0000784,Sita,Hussain,Female,31-03-2008,18,sita.hussain386@rediffmail.com,917836729786,"Apartment 26, Garden Colony, Shimla",CT0061,Shimla,ST007,5845,Tier-3,RG01,Self Employed,626862.0,Mass,Prepaid Mobile,PL002,Retail Store,27-09-2018,90,Active,NaN
1391,C0001392,Rohan,Pradhan,Male,31-03-1980,46,rohan.pradhan706@rediffmail.com,919082312668,"Apartment 19, Lake Phase, Kolkata",CT0005,Kolkata,ST022,700752,Metro,RG04,Auto/Cab Driver,457301.0,Premium,5G,PL013,Online,20-08-2024,19,Active,NaN
19051,C0001392,Rohan,Pradhan,Male,31-03-1980,46,rohan.pradhan706@rediffmail.com,919082312668,"Apartment 19, Lake Phase, Kolkata",CT0005,Kolkata,ST022,700752,Metro,RG04,Auto/Cab Driver,457301.0,Premium,5G,PL013,Online,20-08-2024,19,Active,NaN
1393,C0001394,Preeti,Pawar,Female,31-03-2008,18,preeti.pawar464@yahoo.in,919078014489,"Apartment 300, Tilak Road, Ludhiana",CT0031,Ludhiana,ST003,141305,Tier-2,RG01,Doctor,1263796.0,Value,Postpaid Mobile,PL007,Online,17-02-2022,49,Active,NaN
19020,C0001394,Preeti,Pawar,Female,31-03-2008,18,preeti.pawar464@yahoo.in,919078014489,"Apartment 300, Tilak Road, Ludhiana",CT0031,Ludhiana,ST003,141305,Tier-2,RG01,Doctor,1263796.0,Value,Postpaid Mobile,PL007,Online,17-02-2022,49,Active,NaN


In [17]:
billing = data["billing"]

billing["invoice_id"].duplicated().sum()

np.int64(344)

In [18]:
billing[
    billing["invoice_id"].duplicated(keep=False)
].sort_values("invoice_id").head(20)

,invoice_id,customer_id,billing_date,billing_period_month,base_amount_inr,gst_amount_inr,total_amount_inr,due_date,payment_status
127,INV00000128,C0000030,02-10-2025,10-2025,562.30,101.21,663.51,17-10-2025,Paid
115181,INV00000128,C0000030,02-10-2025,10-2025,562.30,101.21,663.51,17-10-2025,Paid
115278,INV00000652,C0000112,04-07-2025,07-2025,552.95,99.53,652.48,19-07-2025,Overdue
651,INV00000652,C0000112,04-07-2025,07-2025,552.95,99.53,652.48,19-07-2025,Overdue
1219,INV00001220,C0000232,30-01-2026,01-2026,584.68,105.24,689.92,14-02-2026,Overdue
115013,INV00001220,C0000232,30-01-2026,01-2026,584.68,105.24,689.92,14-02-2026,Overdue
1522,INV00001523,C0000272,04-07-2025,07-2025,747.90,134.62,882.52,19-07-2025,Paid
115081,INV00001523,C0000272,04-07-2025,07-2025,747.90,134.62,882.52,19-07-2025,Paid
1599,INV00001600,C0000282,30-01-2026,01-2026,486.59,87.59,574.18,14-02-2026,Overdue
115275,INV00001600,C0000282,30-01-2026,01-2026,486.59,87.59,574.18,14-02-2026,Overdue


### Primary Key / Business Key Validation

The tables identified during the initial duplicate analysis are examined more closely to determine whether their duplicate records also contain repeated key values.

The following keys are checked:

- `customers` → `customer_id`
- `billing` → `invoice_id`
- `payments` → `payment_id`

Comparing complete duplicate rows with duplicate key counts helps determine whether the issue is caused by repeated records or by multiple different records sharing the same key.

The results will be used later when deciding how these records should be cleaned.

In [19]:
key_duplicate_check = pd.DataFrame({
    "table": ["customers", "billing", "payments"],
    "key": ["customer_id", "invoice_id", "payment_id"],
    "duplicate_key_count": [
        data["customers"]["customer_id"].duplicated().sum(),
        data["billing"]["invoice_id"].duplicated().sum(),
        data["payments"]["payment_id"].duplicated().sum()]
})

key_duplicate_check

,table,key,duplicate_key_count
0,customers,customer_id,76
1,billing,invoice_id,344
2,payments,payment_id,276


### Confirm the Nature of the Duplicate Records

A final comparison is performed between complete duplicate rows and duplicate values in the corresponding keys.

If both counts are equal, it indicates that the repeated keys are associated with complete duplicate records rather than different records sharing the same key.

For the affected tables, this comparison will help establish whether the duplicates can later be safely handled by retaining one occurrence of each repeated record.

In [20]:
for table in ["customers", "billing", "payments"]:
    df = data[table]
    
    print(f"{table}:")
    print("  Complete duplicate rows:", df.duplicated().sum())
    print("  Duplicate rows based on key:", df[df.columns[0]].duplicated().sum())
    print()

customers:
  Complete duplicate rows: 76
  Duplicate rows based on key: 76

billing:
  Complete duplicate rows: 344
  Duplicate rows based on key: 344

payments:
  Complete duplicate rows: 276
  Duplicate rows based on key: 276



##     Review the Data Quality Issue Log

The project includes a dedicated `data_quality_issue_log` table containing the intentionally injected data-quality issues and the number of affected records.

This log is used as a reference checklist during validation.

Rather than assuming that every potential data problem exists, the analysis will systematically investigate the issues documented in the log and compare the detected records with the expected `rows_affected` values.

The issue log contains **20 data-quality issues** across the `customers`, `billing`, `payments`, `subscriptions` and `usage_data` tables.

No changes will be made to the issue log because it serves as a validation reference throughout Phase 1.

In [21]:
quality_log = data["data_quality_issue_log"]

quality_log

,table,issue_type,rows_affected
0,customers,missing_email,760
1,customers,missing_address,380
2,customers,missing_income,950
3,customers,invalid_phone,228
4,customers,invalid_email,285
5,customers,incorrect_pin,190
6,customers,misspelled_city,380
7,customers,inconsistent_capitalisation,570
8,customers,impossible_age,38
9,customers,mixed_date_format,380


### Organize the Data Quality Checklist

The quality log is sorted by table and issue type to create a structured checklist for the cleaning process.

This makes it easier to work through the issues systematically, beginning with the customer data and then moving to billing, payments, subscriptions and usage data.

The `rows_affected` column provides an expected benchmark that can be compared with the results of our independent validation checks.

In [22]:
quality_log.sort_values(
    ["table", "issue_type"]
).reset_index(drop=True)

,table,issue_type,rows_affected
0,billing,duplicate_invoices,344
1,billing,future_dates,229
2,billing,incorrect_gst,1149
3,billing,negative_invoice_amount,229
4,customers,duplicate_customers,76
5,customers,impossible_age,38
6,customers,inconsistent_capitalisation,570
7,customers,incorrect_pin,190
8,customers,invalid_email,285
9,customers,invalid_phone,228


### Validate Customer Age

The `age` column is examined for impossible or inconsistent values.

The initial validation checks the statistical range of the column and identifies values that fall outside a reasonable age range.

The data-quality issue log reports **38 records affected by the `impossible_age` issue**, so the validation results will be compared against this benchmark.

The objective is to identify the affected records without modifying the original data at this stage.

In [23]:
customers["age"].describe()

count    19076.000000
mean        36.069721
std         12.125744
min         -5.000000
25%         28.000000
50%         36.000000
75%         44.000000
max        220.000000
Name: age, dtype: float64

### Data Quality Investigation: Validating Customer Age

During the validation of the `age` column, the initial check identified only 23 records with obviously impossible values such as `-5`, `130`, and `220`. However, the `data_quality_issue_log` indicated that **38 records** were intentionally affected by the `impossible_age` issue.

To investigate this discrepancy, the `date_of_birth` column was converted from text into a proper datetime format using `pd.to_datetime()`. The customer's age was then independently recalculated from the year of birth.

An initial attempt used an exact age calculation based on whether the customer's birthday had already occurred by the reference date (31-03-2026). This produced 4,491 apparent age mismatches. Further investigation showed that these were **not data-quality errors**. The dataset's existing `age` values follow a simpler calculation:

> **Age = 2026 − Year of Birth**

For example, a customer born on `01-04-1991` has a recorded age of `35`, even though their birthday falls one day after the reference date. Therefore, the dataset does not use an exact birthday-based age calculation.

After aligning the validation logic with the calculation convention used in the dataset, the comparison identified exactly **38 age mismatches**, matching the `impossible_age` count recorded in the data-quality issue log.

**Conclusion:** The 38 mismatched records are confirmed as the intentionally injected `impossible_age` issues. This investigation highlights the importance of understanding the dataset's underlying calculation logic before classifying records as erroneous.

In [24]:
customers[
    (customers["age"] < 0) | (customers["age"] > 100)
][["customer_id", "date_of_birth", "age"]]

,customer_id,date_of_birth,age
751,C0000752,31-03-1989,220
834,C0000835,31-03-1988,220
2917,C0002918,31-03-1984,130
3026,C0003027,01-04-1971,220
3822,C0003823,01-04-1983,220
3824,C0003825,01-04-1979,220
4639,C0004640,31-03-1982,130
5159,C0005160,31-03-1969,130
5290,C0005291,31-03-2006,130
5833,C0005834,31-03-1982,-5


In [25]:
invalid_age = customers[
    (customers["age"] < 0) | (customers["age"] > 100)
]

invalid_age.shape[0]

23

In [26]:
invalid_age["age"].value_counts().sort_index()

age
-5      6
 130    8
 220    9
Name: count, dtype: int64

In [27]:
customers["date_of_birth_parsed"] = pd.to_datetime(
    customers["date_of_birth"],
    dayfirst=True,
    errors="coerce"
)

In [28]:
customers[["date_of_birth", "date_of_birth_parsed"]].head()

,date_of_birth,date_of_birth_parsed
0,31-03-1988,1988-03-31
1,31-03-1974,1974-03-31
2,31-03-1978,1978-03-31
3,31-03-1988,1988-03-31
4,31-03-1988,1988-03-31


In [29]:
reference_date = pd.Timestamp("2026-03-31")

customers["calculated_age"] = (
    reference_date.year - customers["date_of_birth_parsed"].dt.year
)

In [30]:
reference_date = pd.Timestamp("2026-03-31")

customers["calculated_age"] = (
    reference_date.year
    - customers["date_of_birth_parsed"].dt.year
    - (
        (reference_date.month < customers["date_of_birth_parsed"].dt.month)
        |
        (
            (reference_date.month == customers["date_of_birth_parsed"].dt.month)
            &
            (reference_date.day < customers["date_of_birth_parsed"].dt.day)
        )
    ).astype(int)
)    

In [31]:
age_mismatch = customers[
    customers["age"] != customers["calculated_age"]
][
    ["customer_id", "date_of_birth", "age", "calculated_age"]
]

age_mismatch.shape[0]

4491

In [32]:
customers[["date_of_birth", "age"]].head(20)

,date_of_birth,age
0,31-03-1988,38
1,31-03-1974,52
2,31-03-1978,48
3,31-03-1988,38
4,31-03-1988,38
5,31-03-1990,36
6,31-03-2000,26
7,01-04-1991,35
8,31-03-1996,30
9,31-03-2006,20


In [33]:
customers["date_of_birth_parsed"].min(), customers["date_of_birth_parsed"].max()

(Timestamp('1947-04-01 00:00:00'), Timestamp('2008-03-31 00:00:00'))

In [34]:
customers[customers["age"].isin([-5, 130, 220])][
    ["customer_id", "date_of_birth", "age"]
].head(20)

,customer_id,date_of_birth,age
751,C0000752,31-03-1989,220
834,C0000835,31-03-1988,220
2917,C0002918,31-03-1984,130
3026,C0003027,01-04-1971,220
3822,C0003823,01-04-1983,220
3824,C0003825,01-04-1979,220
4639,C0004640,31-03-1982,130
5159,C0005160,31-03-1969,130
5290,C0005291,31-03-2006,130
5833,C0005834,31-03-1982,-5


In [35]:
age_difference = customers["age"] - customers["calculated_age"]

age_difference.value_counts().sort_index()

-50         1
-49         1
-48         1
-46         1
-43         1
-40         2
-39         1
-35         3
-33         1
-32         2
-29         2
-27         1
-24         1
-23         1
-18         1
-17         1
 0      14585
 1       4453
 73         1
 74         1
 86         1
 88         2
 108        1
 110        1
 112        1
 166        1
 168        1
 170        1
 174        1
 178        1
 182        1
 183        1
 195        2
Name: count, dtype: int64

In [36]:
customers[
    customers["age"] != customers["calculated_age"]
][
    ["customer_id", "date_of_birth", "age", "calculated_age"]
].head(30)

,customer_id,date_of_birth,age,calculated_age
7,C0000008,01-04-1991,35,34
10,C0000011,01-04-1979,47,46
13,C0000014,01-04-1983,43,42
14,C0000015,01-04-1991,35,34
15,C0000016,01-04-1983,43,42
16,C0000017,01-04-2003,23,22
18,C0000019,01-04-1987,39,38
22,C0000023,01-04-1999,27,26
24,C0000025,01-04-2007,19,18
27,C0000028,01-04-2003,23,22


In [37]:
customers["date_of_birth"].value_counts().head(20)

date_of_birth
31-03-2008    1465
31-03-1989     653
31-03-1990     648
31-03-1988     645
01-04-1987     638
31-03-1992     629
31-03-1993     627
31-03-1994     606
31-03-1985     600
01-04-1995     583
31-03-1986     579
01-04-1991     575
01-04-1983     567
31-03-1997     542
31-03-1996     530
31-03-1984     525
31-03-1998     512
31-03-1981     495
31-03-1982     488
31-03-2000     475
Name: count, dtype: int64

In [38]:
customers["calculated_age"] = (
    reference_date.year
    - customers["date_of_birth_parsed"].dt.year
)

In [39]:
age_mismatch = customers[
    customers["age"] != customers["calculated_age"]
][
    ["customer_id", "date_of_birth", "age", "calculated_age"]
]

age_mismatch.shape[0]

38

### Inspect the Affected Age Records

The validation step has identified 38 customer records where the recorded age does not match the age calculated from the customer's year of birth.

Before correcting these records, the affected rows are inspected to understand the nature of the errors and determine whether the correct age can be reliably reconstructed from the available `date_of_birth` information.

No values will be modified during this inspection.

The affected records show clearly invalid recorded ages such as `-5`, `0`, `1`, `130`, and `220`.

And since the correct age can be reconstructed from the customer's date of birth using the dataset's established age calculation logic, the calculated ages will be checked before applying the correction.

No original values will be overwritten until this validation is complete.

In [40]:
age_mismatch.sort_values("age")[[
    "customer_id",
    "date_of_birth",
    "age",
    "calculated_age"
]]

,customer_id,date_of_birth,age,calculated_age
7018,C0007019,31-03-1996,-5,30
5833,C0005834,31-03-1982,-5,44
15136,C0015137,31-03-2004,-5,22
8666,C0008667,31-03-2002,-5,24
13762,C0013763,01-04-1987,-5,39
13922,C0013923,31-03-1981,-5,45
2175,C0002176,01-04-1979,0,47
1604,C0001605,31-03-2002,0,24
15287,C0015288,31-03-2008,0,18
9326,C0009327,01-04-2007,1,19


In [41]:
age_mismatch[[
    "customer_id",
    "date_of_birth",
    "age",
    "calculated_age"
]].sort_values("customer_id")

,customer_id,date_of_birth,age,calculated_age
506,C0000507,31-03-1992,1,34
751,C0000752,31-03-1989,220,37
834,C0000835,31-03-1988,220,38
868,C0000869,31-03-1990,1,36
1604,C0001605,31-03-2002,0,24
2175,C0002176,01-04-1979,0,47
2917,C0002918,31-03-1984,130,42
3026,C0003027,01-04-1971,220,55
3822,C0003823,01-04-1983,220,43
3824,C0003825,01-04-1979,220,47


In [42]:
customers.loc[
    age_mismatch.index,
    "age"
] = customers.loc[
    age_mismatch.index,
    "calculated_age"
]

In [43]:
age_mismatch_after = customers[
    customers["age"] != customers["calculated_age"]
][
    ["customer_id", "date_of_birth", "age", "calculated_age"]
]

age_mismatch_after.shape[0]

0

## Customer Contact Information Validation

The customer age validation has been completed successfully. The next step is to validate the quality of customer contact information.

The `customers` table contains email and phone fields that may contain intentionally invalid values. These fields will be checked using format-based validation rules.

Missing values will be treated separately from invalid formats because a missing contact detail is different from a contact detail that is present but incorrectly formatted.

No values will be modified during this validation step.

In [44]:
email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

invalid_email_mask = (
    customers["email"].notna()
    & ~customers["email"].astype(str).str.match(email_pattern, na=False)
)

invalid_emails = customers.loc[
    invalid_email_mask,
    ["customer_id", "email"]
]

invalid_emails.shape[0]

286

### Inspect Invalid Email Records

The format validation identified 286 non-missing email addresses that do not match the defined email pattern.

The project quality log identifies 285 intentionally invalid email records. Therefore, the additional record requires investigation before any correction is made.

The invalid email values will be inspected to determine whether the difference is caused by the validation rule or by an additional data-quality issue.

No values will be modified during this inspection.

In [45]:
invalid_emails.head(30)

,customer_id,email
203,C0000204,user544_at_mail
268,C0000269,user123_at_mail
481,C0000482,user615_at_mail
717,C0000718,user428_at_mail
822,C0000823,user690_at_mail
829,C0000830,user648_at_mail
830,C0000831,user521_at_mail
857,C0000858,user564_at_mail
869,C0000870,user731_at_mail
998,C0000999,user902_at_mail


In [46]:
invalid_emails.tail(30)

,customer_id,email
16830,C0016831,user612_at_mail
16931,C0016932,user439_at_mail
16986,C0016987,user101_at_mail
17007,C0017008,user573_at_mail
17071,C0017072,user287_at_mail
17096,C0017097,user161_at_mail
17170,C0017171,user837_at_mail
17286,C0017287,user570_at_mail
17407,C0017408,user681_at_mail
17418,C0017419,user430_at_mail


In [47]:
intentional_invalid_pattern = r"^user\d+_at_mail$"

intentional_invalid_mask = invalid_emails["email"].str.match(
    intentional_invalid_pattern,
    na=False
)

intentional_invalid_emails = invalid_emails[
    intentional_invalid_mask
]

intentional_invalid_emails.shape[0]

286

In [48]:
invalid_emails[
    ~intentional_invalid_mask
]

,customer_id,email


In [49]:
invalid_emails["customer_id"].duplicated().sum()

np.int64(1)

In [50]:
invalid_emails[
    invalid_emails["customer_id"].duplicated(keep=False)
]

,customer_id,email
2300,C0002301,user879_at_mail
19062,C0002301,user879_at_mail


In [51]:
invalid_emails["customer_id"].nunique()

285

In [52]:
customers.loc[
    invalid_email_mask,
    "email"
] = pd.NA

In [53]:
invalid_email_mask_after = (
    customers["email"].notna()
    & ~customers["email"].astype(str).str.match(
        email_pattern,
        na=False
    )
)

invalid_email_mask_after.sum()

np.int64(0)

##### AFTER SUCCESSFULLY CONVERTING THE INVALID EMAIL FORMATS WITH MISSING VALUE (NaN) IN ITS FIELD, WE WILL NOW CALCULATE THE TOTAL NO. OF MISSING EMAIL VALUES.

In [54]:
customers["email"].isna().sum()

np.int64(1031)

In [55]:
customers[
    customers["email"].isna()
]["customer_id"].duplicated().sum()

np.int64(2)

In [56]:
customers[
    customers["email"].isna()
]["customer_id"].nunique()

1029

In [57]:
customers[
    customers["customer_id"].duplicated(keep=False)
][[
    "customer_id",
    "email"
]].sort_values("customer_id")

,customer_id,email
97,C0000098,ayaan.sheikh13@rediffmail.com
19044,C0000098,ayaan.sheikh13@rediffmail.com
19074,C0000169,yash.shinde18@outlook.com
168,C0000169,yash.shinde18@outlook.com
19025,C0000784,sita.hussain386@rediffmail.com
783,C0000784,sita.hussain386@rediffmail.com
1391,C0001392,rohan.pradhan706@rediffmail.com
19051,C0001392,rohan.pradhan706@rediffmail.com
1393,C0001394,preeti.pawar464@yahoo.in
19020,C0001394,preeti.pawar464@yahoo.in


In [58]:
customers["customer_id"].duplicated().sum()

np.int64(76)

In [59]:
original_customers = pd.read_csv(
    raw_data_path / "customers.csv"
)

original_customers["email"].isna().sum()

np.int64(745)

In [60]:
original_customers["email"].isna().sum(), quality_log[
    (quality_log["table"] == "customers") &
    (quality_log["issue_type"] == "missing_email")
]["rows_affected"].iloc[0]

(np.int64(745), np.int64(760))

In [61]:
original_customers[
    original_customers["email"].isna()
].shape[0]

745

In [62]:
original_customers[
    original_customers["email"].isna()
]["customer_id"].duplicated().sum()

np.int64(1)

In [63]:
original_customers[
    original_customers["email"].isna()
]["customer_id"].nunique()

744

In [64]:
customers["email"].isna().sum()

np.int64(1031)

### Email validation and cleaning have been completed.

The original customer data contained 745 missing email values and 286 invalid email values. The invalid email values were replaced with missing values because their correct addresses cannot be reliably reconstructed.

After cleaning, all remaining non-missing email values follow the defined format. The data-quality log reports 760 missing-email records, which differs from the 745 missing rows actually present in the raw CSV. This discrepancy is documented rather than altering valid source data to match the benchmark.

### Customer Phone Number Validation

Now the next customer contact field to validate is `phone`.

The project data-quality log identifies intentionally invalid phone numbers, so the phone values will first be inspected to understand their normal format before defining a validation rule.

Missing phone numbers and incorrectly formatted phone numbers will be treated as separate issues.

No phone values will be modified during the initial inspection.

In [65]:
customers.columns.tolist()

['customer_id',
 'first_name',
 'last_name',
 'gender',
 'date_of_birth',
 'age',
 'email',
 'phone_number',
 'address',
 'city_id',
 'city_name',
 'state_id',
 'pin_code',
 'city_tier',
 'region_id',
 'occupation',
 'annual_income_inr',
 'customer_segment',
 'product_line',
 'plan_id',
 'acquisition_channel',
 'acquisition_date',
 'tenure_months',
 'customer_status',
 'churn_date',
 'date_of_birth_parsed',
 'calculated_age']

In [66]:
customers["phone_number"].head(20)

0     916647481156
1     918116536619
2     918958153134
3     917763920098
4     916787478273
5     916083413392
6     916529964485
7     918039201799
8     918153754834
9     916422921624
10           57059
11    919003843105
12    916911182239
13    918055320949
14    919447514418
15    918384315234
16    918575123117
17    917000186679
18    917399234440
19    918847239376
Name: phone_number, dtype: int64

In [67]:
customers["phone_number"].isna().sum()

np.int64(0)

In [68]:
customers["phone_number"].astype(str).str.len().value_counts().sort_index()

phone_number
3         2
4        16
5       210
12    18848
Name: count, dtype: int64

In [69]:
quality_log[
    (quality_log["table"] == "customers") &
    (quality_log["issue_type"] == "invalid_phone")
]["rows_affected"].iloc[0]

np.int64(228)

In [70]:
phone_as_text = customers["phone_number"].astype(str)

valid_phone_pattern = r"^91[6-9]\d{9}$"

valid_phone_mask = phone_as_text.str.match(
    valid_phone_pattern,
    na=False
)

valid_phone_mask.sum()

np.int64(18848)

In [71]:
(~valid_phone_mask).sum()

np.int64(228)

In [72]:
customers.loc[
    ~valid_phone_mask,
    "phone_number"
] = pd.NA

In [73]:
phone_as_text_after = customers["phone_number"].astype("string")

invalid_phone_mask_after = (
    customers["phone_number"].notna()
    & ~phone_as_text_after.str.match(
        valid_phone_pattern,
        na=False
    )
)

invalid_phone_mask_after.sum()

np.int64(18848)

THE OUTPUT FOR THE ABOVE CELL SHOULD HAVE BEEN ZERO AS WE'RE CALCULATING TOTAL NO. OF INVALID PHONE NUMBERS. THE REASON IT SHOWS 18848 IS:

Originally, phone_number was an int64 column.

When we did:
customers.loc[
    ~valid_phone_mask,
    "phone_number"
] = pd.NA

Pandas could no longer keep the column as a normal integer column because integers cannot contain NA in the traditional int64 type. It therefore converted the column to a floating-point type.

So a valid number such as:
916647481156

may now internally look like:
916647481156.0

When we convert that to text, the regex sees the .0, so it no longer matches the 12-digit pattern.
That's why our validation incorrectly considers the 18,848 valid numbers invalid.

Now let's fix it properly. Since we still have the untouched original_customers DataFrame, we'll restore only the phone column from the original data and immediately convert it to pandas' nullable string type.


In [74]:
customers["phone_number"] = original_customers["phone_number"].astype("string")

In [75]:
phone_as_text = customers["phone_number"]

valid_phone_mask = phone_as_text.str.match(
    valid_phone_pattern,
    na=False
)

In [76]:
valid_phone_mask.sum()

np.int64(18848)

In [77]:
customers.loc[
    ~valid_phone_mask,
    "phone_number"
] = pd.NA

In [78]:
invalid_phone_mask_after = (
    customers["phone_number"].notna()
    & ~customers["phone_number"].str.match(
        valid_phone_pattern,
        na=False
    )
)

invalid_phone_mask_after.sum()

np.int64(0)

In [79]:
customers["phone_number"].isna().sum()

np.int64(228)

THE ABOVE CELLS SHOW THAT THE FINAL FIGURE IS 18,848 VALID PHONE NUMBERS + 228 MISSING/CLEANED PHONE NUMBERS = 19,076 TOTAL ROWS.

AND 0 INVALID PHONE NUMBERS

IMPORTANT NOTE: PHONE NUMBERS SHOULD BE STORED AS STRINGS, NOT NUMERIC VALUES, BECAUSE THEY'RE IDENTIFIERS/FORMATTED TEXT RATHER THAN QUANTITIES.

### Now the next customer field to validate is `pin_code`.

The project data-quality log identifies 190 intentionally incorrect PIN codes. The PIN values will first be inspected to determine their data type and expected format before defining a validation rule.

No PIN values will be modified during the initial inspection.

In [80]:
customers["pin_code"].head(20)

0     492680
1     122535
2     390978
3     440881
4     506582
5     796644
6     769163
7     520224
8     495460
9     560151
10    570685
11    492488
12    390803
13    141342
14    492623
15    799999
16    500975
17    180325
18    395057
19    160830
Name: pin_code, dtype: int64

In [81]:
customers["pin_code"].isna().sum()

np.int64(0)

In [82]:
customers["pin_code"].astype(str).str.len().value_counts().sort_index()

pin_code
3       13
4      178
6    18885
Name: count, dtype: int64

In [83]:
invalid_pin_length = customers[
    customers["pin_code"].astype(str).str.len() != 6
][
    ["customer_id", "pin_code"]
]

invalid_pin_length.shape[0]

191

In [84]:
invalid_pin_length["pin_code"].astype(str).str.len().value_counts().sort_index()

pin_code
3     13
4    178
Name: count, dtype: int64

In [85]:
invalid_pin_length.head(30)

,customer_id,pin_code
45,C0000046,9172
182,C0000183,7123
185,C0000186,7128
360,C0000361,4841
517,C0000518,8109
534,C0000535,2180
586,C0000587,1470
610,C0000611,5514
673,C0000674,9869
679,C0000680,6401


In [86]:
invalid_pin_length["customer_id"].duplicated().sum()

np.int64(1)

In [87]:
invalid_pin_length[
    invalid_pin_length["customer_id"].duplicated(keep=False)
].sort_values("customer_id")

,customer_id,pin_code
783,C0000784,5845
19025,C0000784,5845


In [88]:
quality_log[
    (quality_log["table"] == "customers") &
    (quality_log["issue_type"] == "incorrect_pin")
][["rows_affected"]]

,rows_affected
5,190


In [89]:
three_digit_pins = invalid_pin_length[
    invalid_pin_length["pin_code"].astype(str).str.len() == 3
]

three_digit_pins

,customer_id,pin_code
2050,C0002051,540
2900,C0002901,744
6368,C0006369,368
6476,C0006477,355
7177,C0007178,879
7637,C0007638,828
8442,C0008443,124
8938,C0008939,396
12849,C0012850,989
15002,C0015003,183


In [90]:
customers["pin_code"] = customers["pin_code"].astype("Int64")

customers.loc[
    invalid_pin_length.index,
    "pin_code"
] = pd.NA

In [91]:
invalid_pin_mask_after = (
    customers["pin_code"].notna()
    & (customers["pin_code"].astype(str).str.len() != 6)
)

invalid_pin_mask_after.sum()

np.int64(0)

In [92]:
customers["pin_code"].isna().sum()

np.int64(191)

Next major section: Validate Customer Capitalisation

We've now completed the age, email, phone number, and PIN-code validation/cleaning within the customer table.

The quality log also identifies 570 customer records with inconsistent capitalisation. We'll investigate those next, but we won't change anything until we understand the pattern.

In [93]:
customers[["first_name", "last_name", "gender", "city_name", "occupation"]].head(15)

,first_name,last_name,gender,city_name,occupation
0,Harish,Shetty,Male,Raipur,School Teacher
1,Anjali,Khan,Female,Gurugram,Small Business Owner
2,Arjun,Hegde,Male,Vadodara,Auto/Cab Driver
3,Vidya,Parmar,Female,Nagpur,Operations Manager
4,Arjun,Saxena,Male,Warangal,Retired
5,Akash,Ansari,Male,Aizawl,Government Employee
6,Nikhil,Shah,Male,Rourkela,Freelancer
7,Sanjay,Hussain,Male,Vijayawada,Auto/Cab Driver
8,Harish,Chatterjee,Male,Bilaspur,Self Employed
9,Imran,Saha,Male,Bengaluru,Retail Shopkeeper


In [94]:
text_columns = [
    "first_name",
    "last_name",
    "gender",
    "city_name",
    "occupation",
    "customer_segment",
    "product_line",
    "acquisition_channel",
    "customer_status"
]

for column in text_columns:
    case_variants = customers[column].dropna().astype(str).groupby(
        customers[column].dropna().astype(str).str.lower()
    ).nunique()

    print(f"{column}: {case_variants[case_variants > 1].shape[0]} case-variant groups")

first_name: 0 case-variant groups
last_name: 0 case-variant groups
gender: 0 case-variant groups
city_name: 77 case-variant groups
occupation: 0 case-variant groups
customer_segment: 0 case-variant groups
product_line: 0 case-variant groups
acquisition_channel: 0 case-variant groups
customer_status: 0 case-variant groups


In [95]:
city_case_variants = customers[
    customers["city_name"].notna()
].assign(
    city_lower=lambda x: x["city_name"].astype(str).str.lower()
).groupby("city_lower")["city_name"].nunique()

city_case_variants[city_case_variants > 1].sort_values(ascending=False)

city_lower
agartala              2
agra                  2
ahmedabad             2
aizawl                2
ajmer                 2
amritsar              2
aurangabad            2
belagavi              2
bengaluru             2
bhagalpur             2
bhopaal               2
bhopal                2
bhubaneswar           2
bikaner               2
bilaspur              2
chandigarh            2
chennai               2
coimbaatore           2
coimbatore            2
cuttaack              2
cuttack               2
dehradun              2
durgapur              2
gaaya                 2
gangtok               2
gaya                  2
gurugram              2
guwahati              2
gwalior               2
hubballi              2
hyderaabad            2
hyderabad             2
imphaal               2
imphal                2
indore                2
jabalpur              2
jaipur                2
jammu                 2
jamshedpur            2
jodhpur               2
kanpur                2
kochi

In [96]:
capitalisation_mask = (
    customers["city_name"].notna()
    & customers["city_name"].astype(str).str.lower().isin(
        city_case_variants[city_case_variants > 1].index
    )
)

capitalisation_mask.sum()

np.int64(18738)

In [97]:
quality_log[
    (quality_log["table"] == "customers") &
    (quality_log["issue_type"] == "inconsistent_capitalisation")
][["rows_affected"]]

,rows_affected
7,570


capitalisation_mask.sum() returned 18,738, but the quality log says 570.

That's because our mask selected every customer belonging to a city that has multiple capitalization variants.

For example, if Vadodara appears 240 times and VADODARA appears 8 times, our mask counts all 248 rows, even though only the incorrectly capitalized 8 rows are the issue.

So 18,738 is not the number of bad records. We need to identify the incorrect variant within each city.
We can do that by looking at the frequency of each capitalization variant. The most common form will be treated as the standard form, and the less-common variant(s) will be investigated.

In [98]:
city_variant_counts = (
    customers["city_name"]
    .dropna()
    .astype(str)
    .value_counts()
)

city_variant_counts[
    city_variant_counts.index.str.lower().isin(
        city_case_variants[city_case_variants > 1].index
    )
].sort_values(ascending=False)

city_name
Kolkata               491
Noida                 483
Hyderabad             482
Bengaluru             477
Gurugram              462
Chennai               462
New Delhi             454
Ahmedabad             442
Pune                  442
Mumbai                441
Thiruvananthapuram    351
Madurai               333
Surat                 332
Coimbatore            332
Bhubaneswar           330
Lucknow               327
Jodhpur               327
Ludhiana              325
Mysuru                321
Chandigarh            321
Visakhapatnam         320
Indore                318
Kochi                 317
Guwahati              317
Patna                 315
Varanasi              315
Vadodara              311
Aurangabad            309
Vijayawada            308
Nashik                304
Bhopal                303
Ranchi                302
Kanpur                298
Jaipur                298
Dehradun              296
Nagpur                295
Raipur                286
Amritsar              269
Ga

The above output gives us the pattern we were looking for.

The normal city names appear in standard capitalization, while the intentionally corrupted versions appear as things like:
- BHOPAAL
- LUCKNOWX ← this one is actually a misspelling, not capitalization
- IMPHAAL ← likely another issue to distinguish
- Gaaya ← potentially misspelling/case issue

So we should not simply assume that every uncommon variant is a capitalization error. The project has a separate misspelled_city issue, so we need to separate the two.

In [99]:
capitalisation_error_mask = (
    customers["city_name"].notna()
    & (customers["city_name"] != customers["city_name"].astype(str).str.title())
)

capitalisation_error_mask.sum()

np.int64(571)

- Our capitalization check: 571 rows
- Quality log: 570 rows
- Difference: 1 row

The most likely reason is the duplicate customer record we already identified. Since the duplicate row can also have an uppercase city name, it may be getting counted in both the duplicate issue and capitalization issue.

In [100]:
customers.loc[
    capitalisation_error_mask,
    ["customer_id", "city_name"]
].head(30)

,customer_id,city_name
12,C0000013,VADODARA
66,C0000067,HYDERABAD
79,C0000080,SHILLONG
84,C0000085,PUNE
97,C0000098,SILIGURI
138,C0000139,BENGALURU
167,C0000168,NEW DELHI
230,C0000231,BELAGAVI
233,C0000234,MUMBAI
249,C0000250,RANCHI


In [101]:
duplicate_customer_mask = customers["customer_id"].duplicated(keep=False)

capitalisation_and_duplicate = customers[
    capitalisation_error_mask & duplicate_customer_mask
][
    ["customer_id", "city_name"]
]

capitalisation_and_duplicate

,customer_id,city_name
97,C0000098,SILIGURI
19044,C0000098,SILIGURI


We found the reason for the 571 vs 570 difference.

The duplicate customer C0000098 appears twice, and both rows have SILIGURI. Therefore:
- Capitalization mask counts both rows → 571
- The data-quality log counts the underlying affected customer/issue → 570
- The extra 1 row is the duplicate occurrence.

So our validation is consistent with the dataset's intentional issues. Do not remove the duplicate yet, because duplicate handling is a separate cleaning step.

Next: correct the capitalization

We can safely normalize the affected city names to title case. This will convert examples such as:
- SILIGURI → Siliguri
- VADODARA → Vadodara
- NEW DELHI → New Delhi

In [102]:
customers.loc[
    capitalisation_error_mask,
    "city_name"
] = customers.loc[
    capitalisation_error_mask,
    "city_name"
].astype(str).str.title()

In [103]:
capitalisation_error_mask_after = (
    customers["city_name"].notna()
    & (customers["city_name"] != customers["city_name"].astype(str).str.title())
)

capitalisation_error_mask_after.sum()

np.int64(0)

#### NOW THERE ARE ZERO CAPITALIZATION ERRORS REMAINING. SO CAN MOVE ON TO THE NEXT CUSTOMER DATA-QUALITY ISSUE: missing_address

In [104]:
customers["address"].isna().sum()

np.int64(380)

In [105]:
quality_log[
    (quality_log["table"] == "customers") &
    (quality_log["issue_type"] == "missing_address")
][["rows_affected"]]

,rows_affected
1,380


The results match exactly:
- Actual missing addresses: 380
- Quality-log expected count: 380
- Difference: 0

So the missing address issue is fully validated.

Since missing addresses are intentional data-quality issues, we should not invent replacement addresses. For now, we'll leave them as NaN and document them as missing values. They can be handled later depending on the analysis/dashboard requirement.

Next: validate missing email

We already investigated the email issue earlier, including the intentional invalid-email pattern and the duplicate customer complication. The next clean step is to verify the remaining missing emails against the quality log.

In [106]:
customers["email"].isna().sum()

np.int64(1031)

In [107]:
quality_log[
    (quality_log["table"] == "customers") &
    (quality_log["issue_type"] == "missing_email")
][["rows_affected"]]

,rows_affected
0,760


| Check | Rows |
|---|---:|
| Missing emails in original raw data | **745** |
| Quality log: `missing_email` | **760** |
| Invalid email values we identified | **286** |
| Missing emails now, after converting invalid emails to `NaN` | **1,031** |

The 1,031 is correct for the current customers DataFrame, because we deliberately converted the 286 invalid email values into missing values earlier:

745 original missing + 286 invalid emails = 1,031 missing emails

So do not change anything here.

The interesting part is the 745 vs 760 discrepancy between the raw dataset and the quality log. We already independently verified that the raw file itself contains 745 missing emails, so we should document this as a data-quality-log discrepancy rather than artificially creating 15 missing values.

The cell below confirms the original raw count once more from customers.csv (before our cleaning operations.)

In [108]:
original_customers["email"].isna().sum()

np.int64(745)

### The next customer issue: missing annual income.

The quality log says missing_income = 950. First, validate the raw/current data against it.

In [109]:
customers["annual_income_inr"].isna().sum()

np.int64(954)

In [110]:
quality_log[
    (quality_log["table"] == "customers") &
    (quality_log["issue_type"] == "missing_income")
][["rows_affected"]]

,rows_affected
2,950


Here we have another small discrepancy.

Current result
- Current customers["annual_income_inr"] missing: 954
- Quality log says: 950
- Difference: 4 rows

Before changing anything, we should determine whether those 4 extra missing values already existed in the raw CSV or appeared because of something we've done.

So lets run the below cell against the untouched raw dataset:

In [111]:
original_customers["annual_income_inr"].isna().sum()

np.int64(954)

That confirms the 954 missing income values are already present in the original raw CSV.
Therefore, we do not modify these 954 values. The 4-row difference is a discrepancy between the actual dataset and the quality log, not something caused by our cleaning.

In [112]:
customers[
    customers["annual_income_inr"].isna()
][
    ["customer_id", "age", "occupation", "customer_segment", "city_name"]
].head(20)

,customer_id,age,occupation,customer_segment,city_name
14,C0000015,35,Homemaker,Premium,Raipur
25,C0000026,46,Software Engineer,Value,Noida
70,C0000071,42,Doctor,Mass,New Delhi
147,C0000148,44,Retired,Premium,Pune
152,C0000153,18,Small Business Owner,Premium,New Delhi
164,C0000165,39,School Teacher,Value,Guwahati
165,C0000166,42,Small Business Owner,Premium,Siliguri
206,C0000207,42,Student,Value,Hubballi
226,C0000227,25,Retail Shopkeeper,Premium,Aizaawl
258,C0000259,44,Government Employee,Value,Indore


### Next: Validate Misspelled City Names

The quality log identifies 380 misspelled_city records. Since we already corrected capitalization, we should now check whether the city names actually match the official city list in the cities table.

In [113]:
valid_city_names = set(
    data["cities"]["city_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

invalid_city_names = customers[
    ~customers["city_name"].astype(str).str.strip().str.lower().isin(valid_city_names)
]

invalid_city_names.shape[0]

380

In [114]:
invalid_city_names["city_name"].value_counts()

city_name
Chennaai               16
Vaaranasi              13
Kolkaata               12
Ahmedaabad             11
Mumbaai                11
Bhubaaneswar           11
Bengaaluru             10
New Delhix             10
Naashik                10
Noidaa                  9
Indorex                 9
Amritsaar               9
Jodhpurx                9
Coimbaatore             8
Gurugraam               8
Raanchi                 8
Suraat                  8
Maadurai                8
Hyderaabad              7
Dehraadun               7
Visaakhapatnam          7
Guwaahati               7
Kochix                  7
Thiruvaananthapuram     7
Vijaayawada             7
Punex                   6
Ludhiaana               6
Jaaipur                 6
Chaandigarh             6
Bhopaal                 6
Kaanpur                 5
Naagpur                 5
Shillongx               5
Paatna                  5
Waarangal               5
Raaipur                 5
Agaartala               5
Lucknowx                5
Pa

In [115]:
invalid_city_names["city_name"].nunique()

67

In [116]:
sorted(invalid_city_names["city_name"].unique())

['Agaartala',
 'Agraa',
 'Ahmedaabad',
 'Aizaawl',
 'Ajmerx',
 'Amritsaar',
 'Auraangabad',
 'Belaagavi',
 'Bengaaluru',
 'Bhaagalpur',
 'Bhopaal',
 'Bhubaaneswar',
 'Bikaaner',
 'Chaandigarh',
 'Chennaai',
 'Coimbaatore',
 'Cuttaack',
 'Dehraadun',
 'Durgaapur',
 'Gaangtok',
 'Gaaya',
 'Gurugraam',
 'Guwaahati',
 'Gwaalior',
 'Hubbaalli',
 'Hyderaabad',
 'Imphaal',
 'Indorex',
 'Jaabalpur',
 'Jaaipur',
 'Jaammu',
 'Jaamshedpur',
 'Jodhpurx',
 'Kaanpur',
 'Kochix',
 'Kolkaata',
 'Kollaam',
 'Kotaa',
 'Lucknowx',
 'Ludhiaana',
 'Maadurai',
 'Mumbaai',
 'Mysurux',
 'Naagpur',
 'Naashik',
 'New Delhix',
 'Noidaa',
 'Paanaji',
 'Paatna',
 'Punex',
 'Raaipur',
 'Raanchi',
 'Rourkelaa',
 'Saalem',
 'Shillongx',
 'Shimlaa',
 'Siligurix',
 'Suraat',
 'Thiruvaananthapuram',
 'Tiruchiraappalli',
 'Tirupaati',
 'Udaaipur',
 'Vaadodara',
 'Vaaranasi',
 'Vijaayawada',
 'Visaakhapatnam',
 'Waarangal']

#### We now have the complete list of 67 unique misspelled city names. All of them have an obvious corresponding valid city name, so we can create an explicit correction mapping.

In [117]:
city_correction_map = {
    "Agaartala": "Agartala",
    "Agraa": "Agra",
    "Ahmedaabad": "Ahmedabad",
    "Aizaawl": "Aizawl",
    "Ajmerx": "Ajmer",
    "Amritsaar": "Amritsar",
    "Auraangabad": "Aurangabad",
    "Belaagavi": "Belagavi",
    "Bengaaluru": "Bengaluru",
    "Bhaagalpur": "Bhagalpur",
    "Bhopaal": "Bhopal",
    "Bhubaaneswar": "Bhubaneswar",
    "Bikaaner": "Bikaner",
    "Chaandigarh": "Chandigarh",
    "Chennaai": "Chennai",
    "Coimbaatore": "Coimbatore",
    "Cuttaack": "Cuttack",
    "Dehraadun": "Dehradun",
    "Durgaapur": "Durgapur",
    "Gaangtok": "Gangtok",
    "Gaaya": "Gaya",
    "Gurugraam": "Gurugram",
    "Guwaahati": "Guwahati",
    "Gwaalior": "Gwalior",
    "Hubbaalli": "Hubballi",
    "Hyderaabad": "Hyderabad",
    "Imphaal": "Imphal",
    "Indorex": "Indore",
    "Jaabalpur": "Jabalpur",
    "Jaaipur": "Jaipur",
    "Jaammu": "Jammu",
    "Jaamshedpur": "Jamshedpur",
    "Jodhpurx": "Jodhpur",
    "Kaanpur": "Kanpur",
    "Kochix": "Kochi",
    "Kolkaata": "Kolkata",
    "Kollaam": "Kollam",
    "Kotaa": "Kota",
    "Lucknowx": "Lucknow",
    "Ludhiaana": "Ludhiana",
    "Maadurai": "Madurai",
    "Mumbaai": "Mumbai",
    "Mysurux": "Mysuru",
    "Naagpur": "Nagpur",
    "Naashik": "Nashik",
    "New Delhix": "New Delhi",
    "Noidaa": "Noida",
    "Paanaji": "Panaji",
    "Paatna": "Patna",
    "Punex": "Pune",
    "Raaipur": "Raipur",
    "Raanchi": "Ranchi",
    "Rourkelaa": "Rourkela",
    "Saalem": "Salem",
    "Shillongx": "Shillong",
    "Shimlaa": "Shimla",
    "Siligurix": "Siliguri",
    "Suraat": "Surat",
    "Thiruvaananthapuram": "Thiruvananthapuram",
    "Tiruchiraappalli": "Tiruchirappalli",
    "Tirupaati": "Tirupati",
    "Udaaipur": "Udaipur",
    "Vaadodara": "Vadodara",
    "Vaaranasi": "Varanasi",
    "Vijaayawada": "Vijayawada",
    "Visaakhapatnam": "Visakhapatnam",
    "Waarangal": "Warangal"
}

In [118]:
len(city_correction_map)

67

In [119]:
customers["city_name"] = customers["city_name"].replace(city_correction_map)

In [120]:
invalid_city_names_after = customers[
    customers["city_name"].isin(city_correction_map.keys())
]

invalid_city_names_after.shape[0]

0

In [121]:
customers["city_name"].isin(city_correction_map.keys()).sum()

np.int64(0)

#### Next customer data-quality check: date format consistency. This is also one of the issues explicitly listed in the quality log (mixed_date_format).

The `customers` table contains several date-related columns.

Before standardizing dates, we first inspect the date columns and their current formats so that the original data is not modified without validation.

In [122]:
date_columns = [
    column
    for column in customers.columns
    if "date" in column.lower()
]

date_columns

['date_of_birth', 'acquisition_date', 'churn_date', 'date_of_birth_parsed']

##### date_of_birth_parsed is a helper column we created ourselves earlier for age validation. It is not an original source column, so we should exclude it from the mixed-date-format investigation.

The three actual date columns are:
- date_of_birth
- acquisition_date
- churn_date

In [123]:
for column in ["date_of_birth", "acquisition_date", "churn_date"]:
    print(f"\n{column}")
    print(customers[column].dropna().astype(str).head(20).to_list())


date_of_birth
['31-03-1988', '31-03-1974', '31-03-1978', '31-03-1988', '31-03-1988', '31-03-1990', '31-03-2000', '01-04-1991', '31-03-1996', '31-03-2006', '01-04-1979', '31-03-1997', '31-03-1994', '01-04-1983', '01-04-1991', '01-04-1983', '01-04-2003', '31-03-1994', '01-04-1987', '31-03-1985']

acquisition_date
['15-01-2026', '01-01-2014', '11-11-2015', '2014/10/18', '20-10-2014', '20-06-2018', '13-01-2024', '11-09-2024', '28-10-2014', '01-02-2026', '08-06-2015', '11-08-2018', '25-01-2020', '10-07-2024', '02-01-2014', '02-03-2015', '10-05-2016', '05-02-2014', '09-10-2017', '16-11-2015']

churn_date
['22-03-2026', '22-12-2015', '22-10-2023', '06-07-2025', '22-03-2026', '19-03-2020', '20-01-2021', '10-02-2025', '07-08-2023', '22-11-2022', '06-05-2025', '21-11-2022', '29-05-2018', '30-01-2026', '17-06-2018', '25-07-2022', '13-02-2025', '08-06-2024', '22-09-2020', '09-10-2014']


In [124]:
acquisition_dates = customers["acquisition_date"].dropna().astype(str)

mixed_acquisition_dates = acquisition_dates[
    ~acquisition_dates.str.match(r"^\d{2}-\d{2}-\d{4}$")
]

mixed_acquisition_dates.shape[0]

383

383 ROWS ARE DETECTED, WHILE THE QUALITY LOG SAYS 380 AFFECTED ROWS.

THIS IS SIMILAR TO THE EARLIER ISSUES WE FOUND: THE EXTRA 3 ROWS MAY BE CAUSED BY DUPLICATE CUSTOMER RECORDS. WE SHOULD VERIFY/INVESTIGATE THIS BEFORE CLEANING.

In [125]:
mixed_acquisition_customers = customers[
    customers["acquisition_date"].isin(mixed_acquisition_dates)
]

mixed_acquisition_customers[
    mixed_acquisition_customers["customer_id"].duplicated(keep=False)
][["customer_id", "acquisition_date"]].sort_values("customer_id")

,customer_id,acquisition_date
3475,C0003476,2014/07/13
19058,C0003476,2014/07/13
7027,C0007028,2016/08/09
19067,C0007028,2016/08/09
10959,C0010960,2025/09/01
19022,C0010960,2025/09/01


In [126]:
mixed_date_mask = (
    customers["acquisition_date"].notna()
    & ~customers["acquisition_date"].astype(str).str.match(r"^\d{2}-\d{2}-\d{4}$")
)

customers.loc[mixed_date_mask, "acquisition_date"] = (
    pd.to_datetime(
        customers.loc[mixed_date_mask, "acquisition_date"],
        format="%Y/%m/%d",
        errors="coerce"
    )
    .dt.strftime("%d-%m-%Y")
)

In [127]:
remaining_mixed_dates = customers[
    customers["acquisition_date"].notna()
    & ~customers["acquisition_date"].astype(str).str.match(r"^\d{2}-\d{2}-\d{4}$")
]

remaining_mixed_dates.shape[0]

0

##### Next, we'll move to the billing table. The quality log identifies three billing-related issues:
1)  229 future billing dates
2)  229 negative invoice amounts
3)  1,149 incorrect GST values

We'll start with the future billing dates because dates should be validated before we modify any billing values.

In [128]:
billing.columns.tolist()

['invoice_id',
 'customer_id',
 'billing_date',
 'billing_period_month',
 'base_amount_inr',
 'gst_amount_inr',
 'total_amount_inr',
 'due_date',
 'payment_status']

In [129]:
billing["billing_date_parsed"] = pd.to_datetime(
    billing["billing_date"],
    dayfirst=True,
    errors="coerce"
)

billing["billing_date_parsed"].min(), billing["billing_date_parsed"].max()

(Timestamp('2014-02-01 00:00:00'), Timestamp('2027-04-28 00:00:00'))

##### This tells us the billing data ranges from 2014-02-01 to 2027-04-28.

Since the project data uses 2026 as the reference year, the dates after 2026-03-31 are the injected future billing dates we need to investigate. We should confirm the count before cleaning.

In [130]:
reference_date = pd.Timestamp("2026-03-31")

future_billing = billing[
    billing["billing_date_parsed"] > reference_date
]

future_billing.shape[0]

232

THAT IS ANOTHER SMALL DISCREPANCY: 232 ROWS DETECTED VS 229 ROWS IN THE QUALITY LOG. WE SHOULDN'T ASSUME THE REASON YET.

LET'S CHECK WHETHER SOME OF THOSE 232 FUTURE-DATE ROWS BELONG TO DUPLICATED INVOICES, SIMILAR TO WHAT HAPPENED WITH THE CUSTOMER-LEVEL ISSUES.

In [131]:
future_billing[
    future_billing["invoice_id"].duplicated(keep=False)
][["invoice_id", "customer_id", "billing_date"]].sort_values("invoice_id")

,invoice_id,customer_id,billing_date
9101,INV00009102,C0001598,25-04-2027
115095,INV00009102,C0001598,25-04-2027
15976,INV00015977,C0002735,07-08-2026
115265,INV00015977,C0002735,07-08-2026
23207,INV00023208,C0003873,08-07-2026
115086,INV00023208,C0003873,08-07-2026


In [132]:
billing.loc[
    billing["billing_date_parsed"] > reference_date,
    "billing_date"
] = pd.NA

In [133]:
billing["billing_date_parsed"] = pd.to_datetime(
    billing["billing_date"],
    dayfirst=True,
    errors="coerce"
)

(
    billing["billing_date_parsed"] > reference_date
).sum()

np.int64(0)

NOW WE MOVE ONTO NEGATIVE INVOICE AMOUNTS. THE QUALITY LOG REPORTS 229 NEGATIVE INVOICE AMOUNTS IN THE BILLING TABLE.

In [134]:
negative_invoice_amounts = billing[
    billing["total_amount_inr"] < 0
]

negative_invoice_amounts.shape[0]

229

AS THERE IS NO DISCREPANCY, WE CAN NOW CLEAN THEM.

SINCE A NEGATIVE INVOICE TOTAL IS INVALID AND WE DON'T HAVE A RELIABLE BASIS TO CALCULATE THE INTENDED AMOUNT, WE'LL MARK THOSE VALUES AS MISSING RATHER THAN INVENTING A REPLACEMENT.

In [135]:
billing.loc[
    billing["total_amount_inr"] < 0,
    "total_amount_inr"
] = pd.NA

In [136]:
(
    billing["total_amount_inr"] < 0
).sum()

np.int64(0)

NOW WE MOVE ONTO INCORRECT GST VALUES. 

THE QUALITY LOG REPORTS 1,149 INCORRECT GST VALUES. BEFORE CHANGING THEM, WE'LL DETERMINE THE EXPECTED GST AMOUNT FROM THE BILLING DATA ITSELF. 

So, the cell below Selects the three columns needed to understand the billing calculation.

base_amount_inr → amount before GST.

gst_amount_inr → GST charged.

total_amount_inr → final invoice amount.

As we first need to establish the dataset's actual GST calculation pattern.

In [137]:
billing[[
    "base_amount_inr",
    "gst_amount_inr",
    "total_amount_inr"
]].head(10)

,base_amount_inr,gst_amount_inr,total_amount_inr
0,1490.77,268.34,1759.11
1,1396.28,251.33,1647.61
2,1427.24,256.90,1684.14
3,1528.12,275.06,1803.18
4,1502.59,270.47,1773.06
5,1433.41,258.01,1691.42
6,1646.83,296.43,1943.26
7,1637.41,294.73,1932.14
8,1505.66,271.02,1776.68
9,1528.94,275.21,1804.15


THE SAMPLE MAKES THE PATTERN CLEAR: GST APPEARS TO BE 18% OF BASE_AMOUNT_INR, AND TOTAL_AMOUNT_INR IS THE SUM OF BASE + GST.

FOR EXAMPLE, THE FIRST ROW:
- BASE = ₹1,490.77
- 18% GST ≈ ₹268.34
- TOTAL = ₹1,490.77 + ₹268.34 = ₹1,759.11

BEFORE CORRECTING ANYTHING, LET'S VERIFY THIS PATTERN ACROSS THE ENTIRE TABLE.

In [138]:
expected_gst = (billing["base_amount_inr"] * 0.18).round(2)

gst_mismatch = billing[
    billing["gst_amount_inr"].round(2) != expected_gst
]

gst_mismatch.shape[0]

1150

THE EXPECTED OUTPUT WAS 1149 BUT WE GOT 1150, SO LET'S INVESTIGATE WHAT IS THE REASON BEHIND THIS DISCREPANCY

In [139]:
gst_mismatch[
    gst_mismatch["invoice_id"].duplicated(keep=False)
][[
    "invoice_id",
    "base_amount_inr",
    "gst_amount_inr",
    "total_amount_inr"
]].sort_values("invoice_id")

,invoice_id,base_amount_inr,gst_amount_inr,total_amount_inr
18612,INV00018613,361.04,68.68,426.03
115041,INV00018613,361.04,68.68,426.03


In [140]:
gst_mismatch["invoice_id"].nunique()

1149

1,149 UNIQUE INVOICE IDS EXACTLY MATCHES THE QUALITY LOG, SO WE'VE CONFIRMED THE DISCREPANCY WAS CAUSED BY THE DUPLICATED INVOICE.

NOW WE CAN CORRECT THE GST VALUES USING THE VALIDATED 18% OF BASE AMOUNT RULE.

In [141]:
billing.loc[
    gst_mismatch.index,
    "gst_amount_inr"
] = expected_gst.loc[gst_mismatch.index]

In [142]:
expected_gst = (billing["base_amount_inr"] * 0.18).round(2)

(
    billing["gst_amount_inr"].round(2) != expected_gst
).sum()

np.int64(0)

BEFORE MOVING ON, WE SHOULD VALIDATE THE FINAL INVOICE TOTAL BECAUSE WE HAVE NOW CORRECTED BOTH COMPONENTS THAT DETERMINE IT: BASE AMOUNT + GST.

In [143]:
expected_total = (
    billing["base_amount_inr"] + billing["gst_amount_inr"]
).round(2)

total_mismatch = billing[
    billing["total_amount_inr"].round(2) != expected_total
]

total_mismatch.shape[0]

229

229 is exactly what we should expect here. These are the same 229 invoices whose total_amount_inr we intentionally changed to pd.NA because the original totals were negative.

Because a missing value (pd.NA) cannot equal the calculated total, those 229 rows are being counted as mismatches.

We should therefore validate only the non-missing totals.

In [144]:
valid_total_mismatch = billing[
    billing["total_amount_inr"].notna()
    & (billing["total_amount_inr"].round(2) != expected_total)
]

valid_total_mismatch.shape[0]

0

WE'VE NOW COMPLETED ALL THREE BILLING ISSUES FROM THE QUALITY LOG:
- FUTURE BILLING DATES → CLEANED
- NEGATIVE INVOICE TOTALS → MARKED MISSING
- INCORRECT GST → CORRECTED TO 18%
- FINAL INVOICE-TOTAL CONSISTENCY → VERIFIED

NOW, NEXT UP IS PAYMENTS VALIDATION

THE QUALITY LOG IDENTIFIES TWO PAYMENT-RELATED ISSUES:
- 276 DUPLICATE PAYMENTS — WE'VE ALREADY IDENTIFIED THESE DURING THE DUPLICATE-KEY CHECK.
- 184 BROKEN FOREIGN KEYS — SOME PAYMENTS REFERENCE INVOICES THAT DON'T EXIST IN BILLING.

WE'LL NOW INVESTIGATE THE BROKEN FOREIGN KEYS.

In [145]:
payments = data["payments"]

payments.columns.tolist()

['payment_id',
 'invoice_id',
 'customer_id',
 'payment_date',
 'amount_inr',
 'payment_method',
 'payment_status']

NOW WE'LL COMPARE EVERY PAYMENT'S INVOICE_ID AGAINST THE EXISTING INVOICE IDS IN BILLING.

In [146]:
valid_invoice_ids = set(billing["invoice_id"])

broken_payment_fk = payments[
    ~payments["invoice_id"].isin(valid_invoice_ids)
]

broken_payment_fk.shape[0]

184

184 BROKEN PAYMENT → INVOICE REFERENCES EXACTLY MATCH THE QUALITY LOG.

BECAUSE THESE INVOICE_ID VALUES DON'T EXIST IN THE BILLING TABLE, WE SHOULD NOT INVENT REPLACEMENT INVOICE IDS. WE'LL MARK THOSE FOREIGN-KEY VALUES AS MISSING WHILE PRESERVING THE PAYMENT RECORDS THEMSELVES.

In [147]:
payments.loc[
    ~payments["invoice_id"].isin(valid_invoice_ids),
    "invoice_id"
] = pd.NA

In [148]:
remaining_broken_fk = payments[
    payments["invoice_id"].notna()
    & ~payments["invoice_id"].isin(valid_invoice_ids)
]

remaining_broken_fk.shape[0]

0

NOW LETS MOVE ONTO THE 276 DUPLICATE PAYMENTS

In [149]:
payments_before = len(payments)

payments = payments.drop_duplicates(
    subset=["payment_id"],
    keep="first"
)

payments_after = len(payments)

payments_before - payments_after

276

SINCE THESE ARE COMPLETE DUPLICATE PAYMENT RECORDS, WE CAN REMOVE THE DUPLICATE ROWS WHILE KEEPING THE FIRST OCCURRENCE.

 payments_before - payments_after → TELLS US EXACTLY HOW MANY DUPLICATE ROWS WERE REMOVED.

NEXT IS SUBSCRIPTIONS TABLE. THE QUALITY LOG IDENTIFIES TWO ISSUES IN SUBSCRIPTIONS:
- 41 NULL FOREIGN KEYS (PLAN_ID)
- 102 INVALID SUBSCRIPTION CATEGORIES

WE'LL START WITH THE MISSING PLAN_ID VALUES AND VALIDATE THEM AGAINST THE PLANS TABLE.

In [150]:
subscriptions = data["subscriptions"]

subscriptions["plan_id"].isna().sum()

np.int64(41)

WE SHOULD NOW VERIFY WHETHER THE NON-NULL PLAN_ID VALUES ACTUALLY EXIST IN THE PLANS TABLE BEFORE DECIDING WHAT TO DO WITH THE 41 MISSING ONES.
THIS WILL TELL US WHETHER THERE ARE ADDITIONAL BROKEN PLAN REFERENCES BEYOND THE 41 MISSING ONES.

In [151]:
valid_plan_ids = set(data["plans"]["plan_id"])

invalid_subscription_plans = subscriptions[
    subscriptions["plan_id"].notna()
    & ~subscriptions["plan_id"].isin(valid_plan_ids)
]

invalid_subscription_plans.shape[0]

0

0 MEANS THERE ARE NO INVALID NON-MISSING PLAN_ID REFERENCES. SO THE ONLY SUBSCRIPTION PLAN ISSUE IS THE 41 MISSING PLAN_ID VALUES RECORDED IN THE QUALITY LOG.

WE SHOULDN'T INVENT PLAN IDS FOR THOSE 41 SUBSCRIPTIONS. WE'LL LEAVE THEM AS MISSING AND VERIFY THAT ALL REMAINING IDS ARE VALID.

In [152]:
subscription_plan_check = subscriptions[
    subscriptions["plan_id"].notna()
    & ~subscriptions["plan_id"].isin(valid_plan_ids)
]

subscription_plan_check.shape[0]

0

NEXT, WE'LL INVESTIGATE THE 102 INVALID SUBSCRIPTION CATEGORIES, WHICH IS THE SECOND SUBSCRIPTION ISSUE FROM THE QUALITY LOG.

In [153]:
subscriptions.columns.tolist()

['subscription_id',
 'customer_id',
 'plan_id',
 'start_date',
 'end_date',
 'billing_cycle',
 'monthly_charge_inr',
 'subscription_status']

In [154]:
subscriptions["subscription_status"].value_counts(dropna=False)

subscription_status
Active           16930
Terminated        3491
UNKNOWN_STATE      102
Name: count, dtype: int64

THESE 120 UNKNOWN_STATE ARE THE 120 INVALID VALUES

SINCE UNKNOWN_STATE DOESN'T TELL US WHETHER THE SUBSCRIPTION IS ACTUALLY ACTIVE OR TERMINATED, WE SHOULD NOT GUESS. WE'LL CONVERT THEM TO MISSING VALUES.

In [155]:
subscriptions.loc[
    subscriptions["subscription_status"] == "UNKNOWN_STATE",
    "subscription_status"
] = pd.NA

In [156]:
(
    subscriptions["subscription_status"] == "UNKNOWN_STATE"
).sum()

np.int64(0)

NEXT, WE'LL HANDLE THE USAGE_DATA TABLE. THE QUALITY LOG SPECIFICALLY IDENTIFIES:
- 171 OUTLIER VALUES IN USAGE_DATA

In [157]:
usage_data = data["usage_data"]

usage_data.columns.tolist()

['usage_id', 'customer_id', 'usage_month', 'data_gb_used', 'data_sessions']

In [158]:
usage_data["data_gb_used"].describe()

count    57000.000000
mean        40.184885
std        332.574130
min          0.240000
25%         11.390000
50%         19.140000
75%         32.630000
max      22172.700708
Name: data_gb_used, dtype: float64

THE ABOVE OUTPUT CLEARLY SHOWS THE OUTLIER PROBLEM: THE TYPICAL USAGE IS AROUND 11–33 GB, WHILE THE MAXIMUM IS 22,172.7 GB, WHICH IS EXTREMELY FAR FROM THE REST OF THE DISTRIBUTION.

WE SHOULDN'T IMMEDIATELY DELETE OR CAP THESE VALUES. FIRST, LET'S IDENTIFY THE ACTUAL INJECTED OUTLIER VALUES AND CONFIRM THE 171 ROWS FROM THE QUALITY LOG.

WE'RE USING 100 GB ONLY AS AN INSPECTION THRESHOLD, NOT DECLARING EVERY VALUE ABOVE 100 GB INVALID YET.


In [159]:
usage_data[
    usage_data["data_gb_used"] > 100
]["data_gb_used"].sort_values(ascending=False).head(20)

15077    22172.700708
23089    21960.635745
10054    18348.446462
34859    17377.228312
26592    15322.701632
29837    13886.727541
53991    13344.522727
56096    13154.906387
45208    12747.455634
19929    12581.380386
23270    12277.077821
39698    11930.873657
54374    11613.963296
56276    11581.147824
11521    11255.473280
23581    11149.140926
30640    10918.748073
18556    10119.351649
26105    10095.216933
44786     9694.676557
Name: data_gb_used, dtype: float64

THIS CONFIRMS THE EXTREME VALUES ARE GENUINE OUTLIERS IN THE DATASET. HOWEVER, >100 GB WAS ONLY AN INSPECTION THRESHOLD, SO WE SHOULDN'T USE 100 GB AS THE CLEANING RULE.

SINCE THE PROJECT SPECIFICALLY IDENTIFIES 171 OUTLIER ROWS, LET'S USE THE STANDARD IQR (INTERQUARTILE RANGE) METHOD AND SEE WHETHER IT IDENTIFIES EXACTLY THOSE 171 ROWS.

In [160]:
Q1 = usage_data["data_gb_used"].quantile(0.25)
Q3 = usage_data["data_gb_used"].quantile(0.75)

IQR = Q3 - Q1

upper_bound = Q3 + 1.5 * IQR

usage_outliers = usage_data[
    usage_data["data_gb_used"] > upper_bound
]

usage_outliers.shape[0]

4480

THE IQR METHOD DETECTED 4,480 ROWS, NOT THE 171 ROWS SPECIFIED IN THE QUALITY LOG.

THAT MEANS WE SHOULD NOT USE THE IQR RULE TO CLEAN THIS COLUMN. IT IS FLAGGING MANY HIGH-BUT-POTENTIALLY-VALID USAGE VALUES IN ADDITION TO THE INTENTIONALLY INJECTED OUTLIERS.

THE QUALITY LOG TELLS US THAT 171 ROWS WERE INTENTIONALLY INJECTED AS OUTLIERS, SO OUR GOAL IS TO IDENTIFY THE PATTERN OF THOSE INJECTED VALUES RATHER THAN BLINDLY REMOVING EVERY STATISTICAL OUTLIER.

In [161]:
usage_data["data_gb_used"].gt(500).sum(), \
usage_data["data_gb_used"].gt(1000).sum(), \
usage_data["data_gb_used"].gt(5000).sum(), \
usage_data["data_gb_used"].gt(10000).sum()

(np.int64(166), np.int64(149), np.int64(54), np.int64(19))

- 500 GB → 166 ROWS
- 1,000 GB → 149 ROWS
- 5,000 GB → 54 ROWS
- 10,000 GB → 19 ROWS

SO 500 GB IS NOT QUITE THE EXACT BOUNDARY, BECAUSE THE QUALITY LOG SAYS THERE ARE 171 INJECTED OUTLIER ROWS.

RATHER THAN GUESSING THE THRESHOLD, LET'S FIND THE EXACT BOUNDARY BETWEEN THE 171 MOST EXTREME VALUES AND THE REMAINING RECORDS.

In [162]:
sorted_usage = usage_data["data_gb_used"].sort_values(ascending=False)

sorted_usage.iloc[165:175]

35820    544.661710
46580    497.511359
1722     437.824105
40763    371.730000
6297     369.990918
46457    356.540000
43750    345.720000
56565    307.643341
34553    298.130000
21941    292.960000
Name: data_gb_used, dtype: float64

LOOKING AT THE SORTED VALUES:
- 171ST-HIGHEST VALUE = 356.54 GB
- 172ND-HIGHEST VALUE = 345.72 GB

THIS SUGGESTS THAT 350 GB IS THE INTENDED BOUNDARY. LET'S VERIFY IT DIRECTLY RATHER THAN ASSUMING.

In [163]:
usage_outliers = usage_data[
    usage_data["data_gb_used"] > 350
]

usage_outliers.shape[0]

171

SINCE THESE 171 VALUES ARE INTENTIONALLY INJECTED OUTLIERS AND WE DON'T HAVE A RELIABLE BASIS TO DETERMINE THEIR TRUE USAGE, WE'LL MARK THEM AS MISSING RATHER THAN REPLACING THEM WITH AN ESTIMATED VALUE.

In [164]:
usage_data.loc[
    usage_data["data_gb_used"] > 350,
    "data_gb_used"
] = pd.NA

In [165]:
(
    usage_data["data_gb_used"] > 350
).sum()

np.int64(0)

THE NEXT MAJOR PHASE 1 AREA IS COMPLAINTS AND SUPPORT TICKETS, PARTICULARLY THE UNRESOLVED RECORDS AND RESOLUTION FIELDS MENTIONED IN THE PRD.

In [166]:
complaints = data["complaints"]

complaints.columns.tolist()

['complaint_id',
 'customer_id',
 'complaint_date',
 'complaint_type',
 'severity',
 'status',
 'resolution_date']

THE COMPLAINTS TABLE HAS THE FIELDS WE NEED, INCLUDING STATUS AND RESOLUTION_DATE.

WE'LL FIRST INSPECT THE COMPLAINT STATUS DISTRIBUTION. THIS IS IMPORTANT BECAUSE THE PRD SPECIFICALLY ASKS US TO CALCULATE THE UNRESOLVED COMPLAINT SHARE, AND WE NEED TO KNOW THE ACTUAL STATUS VALUES BEFORE DEFINING WHAT COUNTS AS UNRESOLVED.

In [167]:
complaints["status"].value_counts(dropna=False)

status
Resolved      9236
Unresolved    7348
Name: count, dtype: int64

NOW WE'LL CHECK WHETHER THE RESOLUTION_DATE VALUES ARE CONSISTENT WITH THESE STATUSES. A RESOLVED COMPLAINT SHOULD HAVE A RESOLUTION DATE, WHILE AN UNRESOLVED COMPLAINT SHOULD GENERALLY HAVE NO RESOLUTION DATE.

IN THE CELL BELOW,
FIRST CONDITION FINDS RESOLVED COMPLAINTS THAT ARE MISSING A RESOLUTION DATE.
SECOND CONDITION FINDS UNRESOLVED COMPLAINTS THAT INCORRECTLY HAVE A RESOLUTION DATE.

In [168]:
resolved_missing_date = complaints[
    (complaints["status"] == "Resolved")
    & (complaints["resolution_date"].isna())
]

unresolved_with_date = complaints[
    (complaints["status"] == "Unresolved")
    & (complaints["resolution_date"].notna())
]

resolved_missing_date.shape[0], unresolved_with_date.shape[0]

(0, 0)

(0, 0) MEANS THE COMPLAINT STATUS AND RESOLUTION-DATE FIELDS ARE COMPLETELY CONSISTENT:
- EVERY RESOLVED COMPLAINT HAS A RESOLUTION DATE.
- EVERY UNRESOLVED COMPLAINT HAS NO RESOLUTION DATE.

SO THERE IS NO CLEANING REQUIRED FOR THIS PARTICULAR CHECK.

THE PRD ALSO REQUIRES REFERENTIAL-INTEGRITY VALIDATION, SO LET'S VERIFY THAT EVERY COMPLAINT BELONGS TO A VALID CUSTOMER.

In [169]:
valid_customer_ids = set(customers["customer_id"])

broken_complaint_fk = complaints[
    ~complaints["customer_id"].isin(valid_customer_ids)
]

broken_complaint_fk.shape[0]

0

LET'S MOVE TO SUPPORT TICKETS

In [170]:
support_tickets = data["support_tickets"]

support_tickets.columns.tolist()

['ticket_id',
 'customer_id',
 'created_date',
 'category',
 'priority',
 'channel',
 'status',
 'first_contact_resolution',
 'assigned_employee_id',
 'resolution_hours',
 'resolution_date']

In [171]:
support_tickets["status"].value_counts(dropna=False)

status
Resolved     18124
Pending       8450
Escalated     2526
Reopened      2390
Name: count, dtype: int64

IN THE BELOW CELL, FIRST CONDITION FINDS RESOLVED TICKETS MISSING EITHER A RESOLUTION DATE OR RESOLUTION TIME.
SECOND CONDITION FINDS NON-RESOLVED TICKETS THAT INCORRECTLY HAVE RESOLUTION INFORMATION.

In [172]:
resolved_missing_resolution = support_tickets[
    (support_tickets["status"] == "Resolved")
    & (
        support_tickets["resolution_date"].isna()
        | support_tickets["resolution_hours"].isna()
    )
]

unresolved_with_resolution = support_tickets[
    (support_tickets["status"] != "Resolved")
    & (
        support_tickets["resolution_date"].notna()
        | support_tickets["resolution_hours"].notna()
    )
]

resolved_missing_resolution.shape[0], unresolved_with_resolution.shape[0]

(0, 0)

In [173]:
valid_customer_ids = set(customers["customer_id"])
valid_employee_ids = set(data["employees"]["employee_id"])

broken_ticket_customer_fk = support_tickets[
    ~support_tickets["customer_id"].isin(valid_customer_ids)
]

broken_ticket_employee_fk = support_tickets[
    ~support_tickets["assigned_employee_id"].isin(valid_employee_ids)
]

broken_ticket_customer_fk.shape[0], broken_ticket_employee_fk.shape[0]

(0, 0)

(0, 0) CONFIRMS THAT ALL SUPPORT TICKETS HAVE VALID REFERENCES TO BOTH CUSTOMERS AND ASSIGNED EMPLOYEES.

NOW THE REMAINING IMPORTANT PHASE 1 TASK IS A BROADER REFERENTIAL-INTEGRITY CHECK ACROSS THE INTERCONNECTED TABLES. THIS IS IMPORTANT BECAUSE EVEN IF A TABLE HAS NO LISTED INJECTED ISSUE, ITS FOREIGN KEYS STILL NEED TO POINT TO VALID PARENT RECORDS.

WE'LL CHECK THE MAJOR RELATIONSHIPS DEFINED BY THE NEXATEL SCHEMA, STARTING WITH BILLING, PAYMENTS AND SUBSCRIPTIONS.

VALIDATE BILLING → CUSTOMER:

In [174]:
billing_customer_fk = billing[
    ~billing["customer_id"].isin(valid_customer_ids)
]

billing_customer_fk.shape[0]

0

CHECKING THE CUSTOMER RELATIONSHIP IN PAYMENTS:

In [175]:
payment_customer_fk = payments[
    payments["customer_id"].notna()
    & ~payments["customer_id"].isin(valid_customer_ids)
]

payment_customer_fk.shape[0]

0

VALIDATE SUBSCRIPTION → CUSTOMER:

In [176]:
subscription_customer_fk = subscriptions[
    subscriptions["customer_id"].notna()
    & ~subscriptions["customer_id"].isin(valid_customer_ids)
]

subscription_customer_fk.shape[0]

0

VALIDATE CONTRACTS, DEVICES & USAGE RELATIONSHIPS

In [177]:
valid_plan_ids = set(data["plans"]["plan_id"])

contract_customer_fk = data["contracts"][
    data["contracts"]["customer_id"].notna()
    & ~data["contracts"]["customer_id"].isin(valid_customer_ids)
]

contract_plan_fk = data["contracts"][
    data["contracts"]["plan_id"].notna()
    & ~data["contracts"]["plan_id"].isin(valid_plan_ids)
]

device_customer_fk = data["devices"][
    data["devices"]["customer_id"].notna()
    & ~data["devices"]["customer_id"].isin(valid_customer_ids)
]

voice_customer_fk = data["usage_voice"][
    data["usage_voice"]["customer_id"].notna()
    & ~data["usage_voice"]["customer_id"].isin(valid_customer_ids)
]

sms_customer_fk = data["usage_sms"][
    data["usage_sms"]["customer_id"].notna()
    & ~data["usage_sms"]["customer_id"].isin(valid_customer_ids)
]

data_customer_fk = usage_data[
    usage_data["customer_id"].notna()
    & ~usage_data["customer_id"].isin(valid_customer_ids)
]

(
    contract_customer_fk.shape[0],
    contract_plan_fk.shape[0],
    device_customer_fk.shape[0],
    voice_customer_fk.shape[0],
    sms_customer_fk.shape[0],
    data_customer_fk.shape[0]
)

(0, 0, 0, 0, 0, 0)

AT THIS POINT, WE'VE COVERED THE MAJOR TRANSACTIONAL RELATIONSHIPS. BUT, WE STILL NEED TO CHECK THE GEOGRAPHICAL, NETWORK, EMPLOYEE, CAMPAIGN, FEEDBACK, AND RETENTION RELATIONSHIPS.

VALIDATE REMAINING MAJOR FOREIGN KEYS:

In [178]:
valid_city_ids = set(data["cities"]["city_id"])
valid_state_ids = set(data["states"]["state_id"])
valid_region_ids = set(data["regions"]["region_id"])
valid_store_ids = set(data["stores"]["store_id"])
valid_campaign_ids = set(data["marketing_campaigns"]["campaign_id"])

IN THE ABOVE CELL,
EACH LINE CREATES A SET OF VALID PRIMARY KEYS FROM THE CORRESPONDING PARENT TABLE.
THESE SETS LET US EFFICIENTLY CHECK WHETHER FOREIGN-KEY VALUES IN OTHER TABLES ACTUALLY EXIST.

In [179]:
customers_city_fk = customers[
    customers["city_id"].notna()
    & ~customers["city_id"].isin(valid_city_ids)
]

customers_state_fk = customers[
    customers["state_id"].notna()
    & ~customers["state_id"].isin(valid_state_ids)
]

customers_region_fk = customers[
    customers["region_id"].notna()
    & ~customers["region_id"].isin(valid_region_ids)
]

network_city_fk = data["network_quality"][
    data["network_quality"]["city_id"].notna()
    & ~data["network_quality"]["city_id"].isin(valid_city_ids)
]

employee_store_fk = data["employees"][
    data["employees"]["store_id"].notna()
    & ~data["employees"]["store_id"].isin(valid_store_ids)
]

employee_region_fk = data["employees"][
    data["employees"]["region_id"].notna()
    & ~data["employees"]["region_id"].isin(valid_region_ids)
]

stores_city_fk = data["stores"][
    data["stores"]["city_id"].notna()
    & ~data["stores"]["city_id"].isin(valid_city_ids)
]

stores_state_fk = data["stores"][
    data["stores"]["state_id"].notna()
    & ~data["stores"]["state_id"].isin(valid_state_ids)
]

stores_region_fk = data["stores"][
    data["stores"]["region_id"].notna()
    & ~data["stores"]["region_id"].isin(valid_region_ids)
]

(
    customers_city_fk.shape[0],
    customers_state_fk.shape[0],
    customers_region_fk.shape[0],
    network_city_fk.shape[0],
    employee_store_fk.shape[0],
    employee_region_fk.shape[0],
    stores_city_fk.shape[0],
    stores_state_fk.shape[0],
    stores_region_fk.shape[0]
)

(0, 0, 0, 0, 0, 0, 0, 0, 0)

WE STILL NEED TO VALIDATE THE RELATIONSHIPS THAT HAVEN'T BEEN CHECKED YET: CITIES, NETWORK QUALITY, RECHARGES, PLAN HISTORY, FEEDBACK, AND RETENTION CAMPAIGNS.

In [180]:
cities_state_fk = data["cities"][
    data["cities"]["state_id"].notna()
    & ~data["cities"]["state_id"].isin(valid_state_ids)
]

cities_region_fk = data["cities"][
    data["cities"]["region_id"].notna()
    & ~data["cities"]["region_id"].isin(valid_region_ids)
]

network_customer_fk = data["network_quality"][
    data["network_quality"]["customer_id"].notna()
    & ~data["network_quality"]["customer_id"].isin(valid_customer_ids)
]

recharge_customer_fk = data["recharges"][
    data["recharges"]["customer_id"].notna()
    & ~data["recharges"]["customer_id"].isin(valid_customer_ids)
]

recharge_plan_fk = data["recharges"][
    data["recharges"]["plan_id"].notna()
    & ~data["recharges"]["plan_id"].isin(valid_plan_ids)
]

plan_history_customer_fk = data["plan_history"][
    data["plan_history"]["customer_id"].notna()
    & ~data["plan_history"]["customer_id"].isin(valid_customer_ids)
]

plan_history_old_plan_fk = data["plan_history"][
    data["plan_history"]["old_plan_id"].notna()
    & ~data["plan_history"]["old_plan_id"].isin(valid_plan_ids)
]

plan_history_new_plan_fk = data["plan_history"][
    data["plan_history"]["new_plan_id"].notna()
    & ~data["plan_history"]["new_plan_id"].isin(valid_plan_ids)
]

feedback_customer_fk = data["customer_feedback"][
    data["customer_feedback"]["customer_id"].notna()
    & ~data["customer_feedback"]["customer_id"].isin(valid_customer_ids)
]

retention_customer_fk = data["retention_campaigns"][
    data["retention_campaigns"]["customer_id"].notna()
    & ~data["retention_campaigns"]["customer_id"].isin(valid_customer_ids)
]

(
    cities_state_fk.shape[0],
    cities_region_fk.shape[0],
    network_customer_fk.shape[0],
    recharge_customer_fk.shape[0],
    recharge_plan_fk.shape[0],
    plan_history_customer_fk.shape[0],
    plan_history_old_plan_fk.shape[0],
    plan_history_new_plan_fk.shape[0],
    feedback_customer_fk.shape[0],
    retention_customer_fk.shape[0]
)

(0, 0, 0, 0, 0, 0, 0, 0, 0, 0)

THE NEXT IMPORTANT TASK IS TO ACTUALLY REMOVE THE DUPLICATE RECORDS THAT WE IDENTIFIED EARLIER.

| Table | Duplicate key records |
|---|---:|
| `customers` | 76 |
| `billing` | 344 |
| `payments` | 276 |

WE ALREADY REMOVED THE 276 DUPLICATE PAYMENTS IN THE PAYMENTS DATAFRAME. NOW WE'LL CLEAN CUSTOMERS AND BILLING.

In [181]:
customer_rows_before = len(customers)

customers = customers.drop_duplicates(
    subset=["customer_id"],
    keep="first"
)

customer_rows_after = len(customers)

customer_rows_before - customer_rows_after

76

In [182]:
billing_rows_before = len(billing)

billing = billing.drop_duplicates(
    subset=["invoice_id"],
    keep="first"
)

billing_rows_after = len(billing)

billing_rows_before - billing_rows_after

344

NOW LETS UPDATE THE MAIN DATA DICTIONARY.

THIS IS IMPORTANT BECAUSE EARLIER WE CREATED A NEW PAYMENTS DATAFRAME WHEN REMOVING ITS DUPLICATES.
THIS MAKES SURE THE CLEANED VERSIONS OF ALL THREE TABLES ARE STORED BACK INSIDE OUR MAIN DATA DICTIONARY.

In [183]:
data["customers"] = customers
data["billing"] = billing
data["payments"] = payments

ONE FINAL DUPLICATE CHECK:

In [184]:
key_duplicate_check_final = pd.DataFrame({
    "table": ["customers", "billing", "payments"],
    "key": ["customer_id", "invoice_id", "payment_id"],
    "duplicate_key_count": [
        data["customers"]["customer_id"].duplicated().sum(),
        data["billing"]["invoice_id"].duplicated().sum(),
        data["payments"]["payment_id"].duplicated().sum()
    ]
})

key_duplicate_check_final

,table,key,duplicate_key_count
0,customers,customer_id,0
1,billing,invoice_id,0
2,payments,payment_id,0


BEFORE WE SAVE ANYTHING AS OUR CLEANED DATASET, WE SHOULD DO ONE FINAL AUDIT OF ALL 24 TABLES.

WE'LL VERIFY:
1. DUPLICATE ROWS
2. MISSING VALUES
3. PRIMARY-KEY UNIQUENESS
4. MAJOR CLEANING ISSUES WE IDENTIFIED
5. THAT OUR CLEANED DATA STILL HAS THE EXPECTED NUMBER OF TABLES

In [185]:
final_duplicate_report = pd.DataFrame([
    {
        "table": name,
        "duplicate_rows": df.duplicated().sum(),
        "duplicate_percentage": round(
            df.duplicated().sum() / len(df) * 100, 3
        )
    }
    for name, df in data.items()
])

final_duplicate_report = (
    final_duplicate_report
    .sort_values("duplicate_rows", ascending=False)
    .reset_index(drop=True)
)

final_duplicate_report

,table,duplicate_rows,duplicate_percentage
0,billing,0,0.0
1,cities,0,0.0
2,complaints,0,0.0
3,contracts,0,0.0
4,customers,0,0.0
5,customer_feedback,0,0.0
6,data_quality_issue_log,0,0.0
7,devices,0,0.0
8,employees,0,0.0
9,marketing_campaigns,0,0.0


NOW WE NEED TO DISTINGUISH BETWEEN EXPECTED MISSING VALUES AND PROBLEMATIC MISSING VALUES.

FOR EXAMPLE:
- customers.churn_date BEING MISSING IS NORMAL FOR CUSTOMERS WHO HAVEN'T CHURNED.
- subscriptions.end_date BEING MISSING IS NORMAL FOR ACTIVE SUBSCRIPTIONS.
- support_tickets.resolution_date BEING MISSING IS NORMAL FOR UNRESOLVED TICKETS.


In [186]:
final_missing_report = pd.concat([
    pd.DataFrame({
        "table": name,
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_percentage": (
            df.isna().sum().values / len(df) * 100
        ).round(2)
    })
    for name, df in data.items()
], ignore_index=True)

final_missing_report = (
    final_missing_report[
        final_missing_report["missing_count"] > 0
    ]
    .sort_values(
        ["missing_count", "table"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

final_missing_report

,table,column,missing_count,missing_percentage
0,subscriptions,end_date,17017,82.92
1,customers,churn_date,15494,81.55
2,support_tickets,resolution_hours,13366,42.45
3,support_tickets,resolution_date,13366,42.45
4,complaints,resolution_date,7348,44.31
5,customers,email,1029,5.42
6,customers,annual_income_inr,950,5.00
7,customers,address,380,2.00
8,billing,billing_date,229,0.20
9,billing,total_amount_inr,229,0.20


We're not automatically filling these missing values. At this stage we're auditing them and deciding whether each missing value is legitimate or represents a data-quality problem.

There are 17 missing-value patterns remaining, and most are explainable by the business logic of the dataset.

| Table / column | Missing | Interpretation |
|---|---:|---|
| `subscriptions.end_date` | 17,017 | Mostly expected for ongoing subscriptions |
| `customers.churn_date` | 15,494 | Expected for customers who have not churned |
| `support_tickets.resolution_hours` | 13,366 | Expected for unresolved tickets |
| `support_tickets.resolution_date` | 13,366 | Expected for unresolved tickets |
| `complaints.resolution_date` | 7,348 | Expected for unresolved complaints |
| `customers.email` | 1,029 | Missing/invalid emails |
| `customers.annual_income_inr` | 950 | Genuine missing income |
| `customers.address` | 380 | Genuine missing address |
| `billing.billing_date` | 229 | Deliberately removed future dates |
| `billing.total_amount_inr` | 229 | Deliberately removed negative amounts |
| `customers.phone_number` | 228 | Invalid phone numbers converted to missing |
| `customers.pin_code` | 190 | Invalid PINs converted to missing |
| `payments.invoice_id` | 184 | Broken foreign keys converted to missing |
| `usage_data.data_gb_used` | 171 | Outliers converted to missing |
| `subscriptions.subscription_status` | 102 | Invalid `UNKNOWN_STATE` converted to missing |
| `subscriptions.plan_id` | 41 | Deliberately missing foreign keys |

Before declaring all of these legitimate, let's make sure that missing end_date and churn_date actually correspond to ongoing/non-churned records.

In [187]:
subscription_missing_end_check = subscriptions[
    subscriptions["end_date"].isna()
]["subscription_status"].value_counts(dropna=False)

customer_missing_churn_check = customers[
    customers["churn_date"].isna()
]["customer_status"].value_counts(dropna=False)

subscription_missing_end_check, customer_missing_churn_check

(subscription_status
 Active    16930
 NaN          87
 Name: count, dtype: int64,
 customer_status
 Active    15494
 Name: count, dtype: int64)

SOME FINAL DATA QUALITY CHECKS:

In [188]:
final_quality_checks = {
    "invalid_age": (
        (customers["age"] < 0) |
        (customers["age"] > 100)
    ).sum(),

    "invalid_email_format": (
        customers["email"]
        .astype("string")
        .str.match(r"^user\d+_at_mail$", na=False)
    ).sum(),

    "invalid_phone_length": (
        ~customers["phone_number"]
        .astype("string")
        .str.match(r"^91[6-9]\d{9}$", na=False)
        & customers["phone_number"].notna()
    ).sum(),

    "invalid_pin_length": (
        customers["pin_code"]
        .astype("string")
        .str.len()
        .ne(6)
        & customers["pin_code"].notna()
    ).sum(),

    "future_billing_dates": (
        billing["billing_date_parsed"] > reference_date
    ).sum(),

    "negative_invoice_amounts": (
        billing["total_amount_inr"] < 0
    ).sum(),

    "invalid_subscription_status": (
        subscriptions["subscription_status"] == "UNKNOWN_STATE"
    ).sum(),

    "extreme_usage_values": (
        usage_data["data_gb_used"] > 350
    ).sum()
}

pd.Series(final_quality_checks, name="remaining_issues")

invalid_age                    0
invalid_email_format           0
invalid_phone_length           0
invalid_pin_length             0
future_billing_dates           0
negative_invoice_amounts       0
invalid_subscription_status    0
extreme_usage_values           0
Name: remaining_issues, dtype: int64

Now lets remove Temporary Helper Columns

During validation, we created three helper columns:
- date_of_birth_parsed
- calculated_age
- billing_date_parsed

These were useful for checking the data, but they are not part of the original dataset schema, so we should remove them before saving the cleaned data.

In [189]:
customers = customers.drop(
    columns=["date_of_birth_parsed", "calculated_age"]
)

billing = billing.drop(
    columns=["billing_date_parsed"]
)

data["customers"] = customers
data["billing"] = billing

In [190]:
print("date_of_birth_parsed" in customers.columns)
print("calculated_age" in customers.columns)
print("billing_date_parsed" in billing.columns)

False
False
False


BEFORE SAVING THE CLEANED DATASETS, LET'S CONFIRM THAT:
1. WE STILL HAVE ALL 24 TABLES.
2. EVERY TABLE STILL HAS COLUMNS.
3. THE CLEANED TABLES ARE STORED IN OUR DATA DICTIONARY.

In [191]:
final_structure_check = pd.DataFrame([
    {
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    }
    for name, df in data.items()
])

final_structure_check = (
    final_structure_check
    .sort_values("table")
    .reset_index(drop=True)
)

print("Number of tables:", len(data))
final_structure_check

Number of tables: 24


,table,rows,columns
0,billing,114969,9
1,cities,68,8
2,complaints,16584,7
3,contracts,10607,9
4,customer_feedback,28499,7
5,customers,19000,25
6,data_quality_issue_log,20,3
7,devices,19000,8
8,employees,800,9
9,marketing_campaigns,40,9


NOW WE'LL SAVE THE CLEANED DATASETS.
ALL VALIDATED AND CLEANED TABLES WILL NOW BE EXPORTED AS CSV FILES INTO THE `data/cleaned` FOLDER. THE ORIGINAL FILES IN `data/raw` WILL REMAIN UNCHANGED.

In [192]:
cleaned_data_path = Path("../data/cleaned")

cleaned_data_path.mkdir(parents=True, exist_ok=True)

for table_name, df in data.items():
    output_file = cleaned_data_path / f"{table_name}.csv"
    df.to_csv(output_file, index=False)

print(f"Saved {len(data)} cleaned tables to: {cleaned_data_path}")

Saved 24 cleaned tables to: ..\data\cleaned


# Data Quality Report — NexaTel Customer Churn Analytics

## 1. Dataset Overview
The NexaTel dataset contains 24 interconnected tables covering customers, subscriptions, billing, payments, usage, network quality, complaints, support, campaigns and geographical information. The raw datasets were loaded and profiled before cleaning. The original files in `data/raw/` were preserved without modification.

## 2. Data Quality Issues Identified

The following issues were identified during the validation process:

- Duplicate customer records and duplicate billing invoices.
- Duplicate payment records.
- Invalid customer ages, including negative and unrealistic values.
- Invalid email formats.
- Invalid phone numbers and PIN codes.
- Inconsistent capitalization in customer city names.
- Misspelled city names.
- Mixed date formats in customer acquisition dates.
- Future billing dates.
- Negative invoice amounts.
- Incorrect GST amounts.
- Broken payment-to-invoice foreign keys.
- Invalid subscription status values.
- Missing subscription plan references.
- Extreme/outlier values in data usage.
- Missing customer address, email and income values.

## 3. Cleaning Actions Performed

The identified issues were cleaned using rule-based validation:

- Removed 76 duplicate customer records.
- Removed 344 duplicate billing invoice records.
- Removed 276 duplicate payment records.
- Invalid ages were corrected using date-of-birth-based validation.
- Invalid email values were converted to missing values.
- Invalid phone numbers and PIN codes were converted to missing values.
- Customer city capitalization was standardized.
- 380 misspelled city records were corrected using the official city reference table.
- Mixed acquisition-date formats were standardized to DD-MM-YYYY.
- 229 future billing dates were converted to missing values.
- 229 negative invoice totals were converted to missing values.
- Incorrect GST values were recalculated using the expected 18% GST relationship.
- 184 broken payment invoice references were converted to missing values.
- 102 invalid subscription status values were converted to missing values.
- 41 missing subscription plan references were retained as missing.
- 171 extreme data-usage values above 350 GB were converted to missing values.

## 4. Missing-Value Assessment

Remaining missing values were reviewed according to business context.

Missing `churn_date` values correspond to active customers, while missing subscription `end_date` values primarily correspond to active subscriptions. Missing support-ticket resolution fields correspond to unresolved tickets, and missing complaint resolution dates correspond to unresolved complaints.

Other missing values, such as customer income, address and email, were retained rather than artificially imputed because there was insufficient evidence to determine their correct values.

## 5. Referential Integrity

Foreign-key validation was performed across the relevant interconnected tables. All tested foreign-key relationships returned zero unresolved invalid references after cleaning.

## 6. Final Validation

The final validation confirmed:

- 24 tables successfully retained.
- 0 duplicate rows across all 24 tables.
- 0 duplicate primary-key values in customers, billing and payments.
- 0 remaining invalid ages.
- 0 remaining invalid email formats.
- 0 remaining invalid phone numbers.
- 0 remaining invalid PIN codes.
- 0 remaining future billing dates.
- 0 remaining negative invoice amounts.
- 0 remaining invalid subscription statuses.
- 0 remaining extreme usage values.

## 7. Data Quality Log Comparison

The `data_quality_issue_log` was used as a reference for intentionally introduced data-quality issues. In several cases, detected row counts differed slightly from the log because duplicate records caused multiple row occurrences for the same affected entity. A few missing-value counts also differed from the log and were retained according to the actual values present in the raw datasets rather than being artificially changed to match the log.

## 8. Conclusion

The NexaTel datasets have been cleaned, validated and saved as 24 cleaned CSV files in `data/cleaned/`. The raw datasets remain unchanged in `data/raw/`. The cleaned datasets are now ready for exploratory data analysis, visualization and churn KPI calculation.

# Phase 1 Complete — Data Cleaning & Quality Validation

The NexaTel datasets have been successfully profiled, cleaned and validated.

### Final Status
- 24/24 tables processed
- 24/24 cleaned CSV files saved
- Duplicate records removed
- Data-quality issues addressed
- Referential integrity validated
- Final quality checks passed
- Raw datasets preserved

